In [2]:
"""
kaggle_train.py  [v15 — CAPACITY UPGRADE TO 160 CHANNELS]
=========================================================================
This script imports the 300k weights from v14 (128 channels) and automatically
resizes them into a 160-channel architecture for the final push to perfection.
"""

import os
import sys
import json
import time
import math
import random
import shutil
import zipfile
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
import torchvision.transforms as T
from tqdm import tqdm

# ============================================================
# KAGGLE PATHS
# ============================================================

KAGGLE_DATA_ROOT = Path("/kaggle/input")
KAGGLE_WORK_DIR  = Path("/kaggle/working")

# ============================================================
# CUSTOM LOADER FOR V15 UPGRADE
# ============================================================
def transfer_weights_with_resize(model, old_state_dict):
    """Slices and copies weights from a smaller model into a larger model."""
    model_state = model.state_dict()
    transferred = 0
    for name, old_param in old_state_dict.items():
        if name in model_state:
            new_param = model_state[name]
            if old_param.shape == new_param.shape:
                new_param.copy_(old_param)
                transferred += 1
            elif old_param.dim() == new_param.dim():
                # Shape mismatch (e.g., 128 -> 160). Copy the overlapping 128 slice.
                slices = tuple(slice(0, min(dim_old, dim_new)) for dim_old, dim_new in zip(old_param.shape, new_param.shape))
                new_param[slices].copy_(old_param[slices])
                transferred += 1
    return transferred

# ============================================================
# RESUME — Copy checkpoint + CSV from input datasets
# ============================================================

_ckpt_dst_dir = KAGGLE_WORK_DIR / "checkpoints"
_ckpt_dst_dir.mkdir(parents=True, exist_ok=True)
_dst_ckpt = _ckpt_dst_dir / "last.pt"

if not _dst_ckpt.exists():
    _candidates = [
        KAGGLE_DATA_ROOT / "datasets/devsach/relight-checkpoint" / "last.pt",
        KAGGLE_DATA_ROOT / "datasets/coding12341234/relight-checkpoint" / "last.pt",
        KAGGLE_DATA_ROOT / "relight-checkpoint" / "last.pt",
    ]
    for _src in _candidates:
        if _src.exists():
            shutil.copy2(_src, _dst_ckpt)
            print(f"  [RESUME] Copied checkpoint to last.pt ✓")
            break

def _is_valid_pt(path):
    try:
        ck        = torch.load(path, map_location="cpu")
        step      = ck.get("step", 0)
        loss      = ck.get("best_loss", float("inf"))
        unet_state = ck.get("unet", {})
        for k, v in list(unet_state.items())[:5]:
            if torch.is_tensor(v) and (torch.isnan(v).any() or torch.isinf(v).any()):
                print(f"    [CORRUPT] NaN/Inf in weights: {k}")
                return False, 0, float("inf")
        return True, step, loss
    except Exception:
        return False, 0, float("inf")

FORCE_FRESH_START = False   

if FORCE_FRESH_START and _dst_ckpt.exists():
    _dst_ckpt.unlink()
    print("  [v15] FRESH START — old checkpoint deleted ✓")
elif not _dst_ckpt.exists():
    print("  [v15] No checkpoint found — will train from scratch ✓")
else:
    valid, step, loss = _is_valid_pt(_dst_ckpt)
    if valid:
        print(f"  [RESUME] last.pt exists  (step={step}, loss={loss:.4f})")
    else:
        _dst_ckpt.unlink()
        print("  [RESUME] Corrupt checkpoint deleted — fresh start")

_csv_src = KAGGLE_DATA_ROOT / "datasets/coding12341234/relight-metadata" / "master_metadata.csv"
_dst_csv = KAGGLE_WORK_DIR / "master_metadata.csv"
if _csv_src.exists() and not _dst_csv.exists():
    shutil.copy2(_csv_src, _dst_csv)
    print(f"  [RESUME] Copied master_metadata.csv ✓")
else:
    print(f"  [RESUME] master_metadata.csv: {'already exists' if _dst_csv.exists() else 'NOT FOUND'}")

# ============================================================
# FIND DATA ROOT
# ============================================================

def find_alexander_root():
    candidates = [
        KAGGLE_DATA_ROOT / "alexander" / "alexander",
        KAGGLE_DATA_ROOT / "alexander",
        KAGGLE_DATA_ROOT / "relight-dataset" / "alexander",
        KAGGLE_DATA_ROOT / "synthlight" / "alexander",
        KAGGLE_DATA_ROOT / "datasets/devsachaniitk/alexander/alexander",
        Path("D:/alexander"),
    ]
    for c in candidates:
        if c.exists():
            print(f"  Found data root: {c}")
            return c
    for p in KAGGLE_DATA_ROOT.rglob("synthlight_renders_v6"):
        return p.parent
    raise FileNotFoundError("Could not find alexander/ folder.")

ALEXANDER_ROOT = find_alexander_root()
HDRI_CACHE_DIR = Path("/kaggle/input/datasets/devsachaniitk/hdri-cache/hdri_cache")
if not HDRI_CACHE_DIR.exists():
    found = list(Path("/kaggle/input").rglob("hdri_cache"))
    HDRI_CACHE_DIR = found[0] if found else HDRI_CACHE_DIR

SH_CACHE_DIR = KAGGLE_WORK_DIR / "sh_cache"
CSV_PATH     = KAGGLE_WORK_DIR / "master_metadata.csv"
CKPT_DIR     = KAGGLE_WORK_DIR / "checkpoints"
SAMPLE_DIR   = KAGGLE_WORK_DIR / "samples"
DIAG_DIR     = KAGGLE_WORK_DIR / "diag_grids"
for d in [SH_CACHE_DIR, CKPT_DIR, SAMPLE_DIR, DIAG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  [DEVICE] {DEVICE}")

# ============================================================
# GPU CONFIG — T4 tuned
# ============================================================

def get_gpu_config():
    if not torch.cuda.is_available():
        return {"tier": "cpu",  "name": "CPU", "batch_size": 2,
                "base_ch": 64,  "use_amp": False, "ddim_steps": 20, "num_workers": 0}
    name = torch.cuda.get_device_name(0).lower()
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    if "a100" in name:
        return {"tier": "a100", "name": name, "batch_size": 8,
                "base_ch": 160, "use_amp": True, "ddim_steps": 50, "num_workers": 4}
    elif "t4" in name or mem < 17:
        return {"tier": "t4",   "name": name, "batch_size": 4,
                "base_ch": 160, "use_amp": True, "ddim_steps": 100, "num_workers": 2}
    else:
        return {"tier": "other","name": name, "batch_size": 6,
                "base_ch": 160, "use_amp": True, "ddim_steps": 100, "num_workers": 4}

GPU_CFG = get_gpu_config()
print(f"  [GPU] {GPU_CFG['tier']} — batch={GPU_CFG['batch_size']} base_ch={GPU_CFG['base_ch']}")

# ============================================================
# CONFIG
# ============================================================

VERSION_TO_SUBJECT = {
    "synthlight_renders_v6"          : 1,
    "synthlight_renders_v7"          : 2,
    "synthlight_renders_v8"          : 3,
}
NUM_SUBJECTS = 3       
SID_EMBED_DIM = 256    

CFG = {
    "device"          : DEVICE,
    "batch_size"      : GPU_CFG["batch_size"],
    "use_amp"         : GPU_CFG["use_amp"],
    "image_size"      : 256,
    "total_steps"     : 500_000,
    "lr"              : 3e-5,
    "lr_min"          : 5e-7,
    "wd"              : 1e-4,
    "T"               : 1000,
    "base_ch"         : 160,
    "ze_dim"          : 256,

    "mse_weight"      : 1.0,
    "lpips_weight"    : 4.0,   # Increased to force sharpness
    "ch_loss_weight"  : 1.5,
    "shadow_weight"   : 12.0,  # Pushed to deepen blacks
    "freq_loss_weight": 3.0,
    "color_mv_weight" : 3.5,   # Relaxed to allow vibrant skin
    "sat_loss_weight" : 3.5,   # Relaxed to allow vibrant skin

    "fg_weight"       : 5.0,
    "bg_weight"       : 0.15,
    "num_workers"     : GPU_CFG["num_workers"],
    "log_every"       : 100,
    "sample_every"    : 1_000,
    "diag_every"      : 10_000,
    "save_last_every" : 500,
    "save_ckpt_every" : 5_000,
    "val_fraction"    : 0.02,
    "accum_steps"     : 2,      
    "ddim_steps"      : GPU_CFG["ddim_steps"],
    "wandb_project"   : "relight-diffusion",
    "lpips_every"     : 1,
    "lpips_max_t"     : 150,
    "val_every"       : 5_000,
    "bg_fill_value"   : -0.06,
}

DEBUG_MODE         = False
DEBUG_STEPS        = 500
DEBUG_SAMPLE_EVERY = 50
DEBUG_DIAG_AT      = 500

# ============================================================
# WANDB
# ============================================================

_wandb = None

def init_wandb():
    global _wandb
    try:
        import wandb
        api_key = os.environ.get("WANDB_API_KEY", "")
        if not api_key:
            try:
                from kaggle_secrets import UserSecretsClient
                api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
            except Exception:
                pass
        if not api_key:
            print("  [WandB] WANDB_API_KEY not found — logging disabled.")
            return
        wandb.login(key=api_key, relogin=True)
        wandb.init(project=CFG["wandb_project"], config=CFG, resume="allow",
                   name="relight-v15-160ch")
        _wandb = wandb
        print("  [WandB] Connected ✓")
    except Exception as e:
        print(f"  [WandB] Failed: {e}")

def wandb_log(d):
    if _wandb:
        try: _wandb.log(d)
        except Exception: pass

def wandb_log_image(key, img, step):
    if _wandb:
        try: _wandb.log({key: _wandb.Image(img)}, step=step)
        except Exception: pass

def wandb_finish():
    if _wandb:
        try: _wandb.finish()
        except Exception: pass

# ============================================================
# DATA HELPERS
# ============================================================

def _normalise_hdri_key(raw: str) -> str:
    key = raw.strip().lower()
    for ext in (".hdr", ".exr", ".png"):
        if key.endswith(ext):
            key = key[:-len(ext)]; break
    return key

def compute_sh_from_npy(hdr_arr, n_samples=50000):
    C, H, W    = hdr_arr.shape
    ys         = (np.arange(H) + 0.5) / H
    sin_theta  = np.sin(ys * np.pi)
    weights    = sin_theta / (sin_theta.sum() + 1e-8)
    flat_w     = np.repeat(weights, W)
    flat_w    /= flat_w.sum()
    flat_r = hdr_arr[0].ravel(); flat_g = hdr_arr[1].ravel(); flat_b = hdr_arr[2].ravel()
    idx = np.random.choice(len(flat_r), size=min(n_samples, len(flat_r)),
                           replace=False, p=flat_w)
    xs_all   = (np.arange(W) + 0.5) / W * 2 * np.pi
    ys_all   = ys * np.pi
    phi_flat = np.tile(xs_all, H); theta_flat = np.repeat(ys_all, W)
    phi = phi_flat[idx]; theta = theta_flat[idx]
    r = flat_r[idx]; g = flat_g[idx]; b = flat_b[idx]
    x = np.sin(theta)*np.cos(phi); y = np.sin(theta)*np.sin(phi); z = np.cos(theta)
    Y = np.stack([
        np.ones(len(x)) * 0.282095,
        y * 0.488603, z * 0.488603, x * 0.488603,
        x*y * 1.092548, y*z * 1.092548,
        (3*z*z-1) * 0.315392, x*z * 1.092548,
        (x*x-y*y) * 0.546274,
    ], axis=1)
    coeffs = np.concatenate([
        (Y * r[:, None]).mean(axis=0),
        (Y * g[:, None]).mean(axis=0),
        (Y * b[:, None]).mean(axis=0),
    ])
    return coeffs.astype(np.float32)

def precompute_sh_if_needed():
    sh_npy  = SH_CACHE_DIR / "sh_coeffs.npy"
    sh_json = SH_CACHE_DIR / "sh_index.json"
    if sh_npy.exists() and sh_json.exists():
        print("  SH cache found — skipping.")
        return str(sh_npy), str(sh_json)

    _sh_input_candidates = [
        KAGGLE_DATA_ROOT / "datasets/coding12341234/relight-sh-cache",
        KAGGLE_DATA_ROOT / "relight-sh-cache",
    ]
    for _sh_src in _sh_input_candidates:
        _sh_npy_src  = _sh_src / "sh_coeffs.npy"
        _sh_json_src = _sh_src / "sh_index.json"
        if _sh_npy_src.exists() and _sh_json_src.exists():
            shutil.copy2(_sh_npy_src,  sh_npy)
            shutil.copy2(_sh_json_src, sh_json)
            print(f"  SH cache loaded from input dataset ✓  ({_sh_src.name})")
            return str(sh_npy), str(sh_json)

    print("  Computing SH coefficients from HDR cache...")
    hdr_files = sorted(HDRI_CACHE_DIR.glob("*_hdr.npy"))
    if not hdr_files:
        raise FileNotFoundError(f"No *_hdr.npy files in {HDRI_CACHE_DIR}")
    all_coeffs = []; sh_index = {}
    for i, fpath in enumerate(hdr_files):
        arr = np.load(fpath)
        if arr.ndim != 3 or arr.shape[0] != 3: continue
        key = _normalise_hdri_key(fpath.stem.replace("_hdr", ""))
        sh_index[key] = len(all_coeffs)
        all_coeffs.append(compute_sh_from_npy(arr))
        if (i+1) % 100 == 0: print(f"    [{i+1}/{len(hdr_files)}]")
    arr_out = np.stack(all_coeffs)
    np.save(sh_npy, arr_out)
    with open(sh_json, "w") as f: json.dump(sh_index, f)
    print(f"  SH done: {arr_out.shape}")
    return str(sh_npy), str(sh_json)

def precompute_latent_cache(vae, subject_masks, device, image_size=256):
    cache_npy  = KAGGLE_WORK_DIR / "latent_cache.npy"
    cache_json = KAGGLE_WORK_DIR / "latent_index.json"

    if cache_npy.exists() and cache_json.exists():
        print("  Latent cache found — loading...")
        arr   = np.load(cache_npy)
        index = json.load(open(cache_json))
        print(f"  Latent cache: {arr.shape[0]:,} entries loaded ✓")
        return torch.from_numpy(arr).float(), index

    import pandas as pd
    print("  Precomputing latent cache (one-time, ~10 min on T4)...")
    df        = pd.read_csv(CSV_PATH)
    transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize([0.5]*3, [0.5]*3),
    ])

    latents = []
    index   = {}
    skipped = 0

    for i, row in df.iterrows():
        path = str(row["image_path"])
        sid  = int(row["subject_id"])
        try:
            img     = transform(Image.open(path).convert("RGB"))
            img     = img.unsqueeze(0).to(device)
            sid_t   = torch.tensor([sid])
            img_m   = apply_fg_mask(img, subject_masks, sid_t)
            z       = vae_encode(vae, img_m).squeeze(0).cpu().numpy()
            index[path] = len(latents)
            latents.append(z.astype(np.float16))
        except Exception:
            skipped += 1

        if (i + 1) % 5000 == 0:
            print(f"    [{i+1:,}/{len(df):,}]  skipped={skipped}")

    arr = np.stack(latents)   
    np.save(cache_npy, arr)
    json.dump(index, open(cache_json, "w"))
    print(f"  Latent cache saved: {arr.shape}  skipped={skipped} ✓")
    return torch.from_numpy(arr).float(), index

def build_csv_if_needed():
    if CSV_PATH.exists():
        print("  CSV found — skipping build.")
        return str(CSV_PATH)

    import csv
    print("  Building master_metadata.csv...")
    rows = []
    for version, sid in VERSION_TO_SUBJECT.items():
        mesh_dir = ALEXANDER_ROOT / version / "model_mesh"
        if not mesh_dir.exists(): continue
        for json_file in sorted(mesh_dir.glob("*.json")):
            with open(json_file) as f:
                entries = json.load(f)
            if isinstance(entries, dict):
                entries = next(v for v in entries.values() if isinstance(v, list))
            for e in entries:
                fname    = e.get("filename", "")
                img_path = ALEXANDER_ROOT / version / "model_mesh" / fname
                if not img_path.exists(): continue
                hdri_name = e.get("hdri_name", "")
                hdri_key  = _normalise_hdri_key(hdri_name)
                rows.append({
                    "image_path"        : str(img_path),
                    "subject_id"        : sid,
                    "filename"          : fname,
                    "hdri_name"         : hdri_name,
                    "hdri_name_key"     : hdri_key,
                    "hdri_rank"         : e.get("hdri_rank", 0),
                    "hdri_tier"         : e.get("hdri_tier", 0),
                    "hdri_tier_name"    : e.get("hdri_tier_name", ""),
                    "hdri_score"        : e.get("hdri_score", 0.0),
                    "hdri_avg_lum"      : e.get("hdri_avg_lum", 0.0),
                    "hdri_peak_lum"     : e.get("hdri_peak_lum", 0.0),
                    "hdri_dyn_range"    : e.get("hdri_dyn_range", 0.0),
                    "hdri_strength"     : e.get("hdri_strength", 1.0),
                    "hdri_saturation"   : e.get("hdri_saturation", 1.0),
                    "hdri_rotation_deg" : e.get("hdri_rotation_deg", 0.0),
                    "hdri_rotation_idx" : e.get("hdri_rotation_idx", 0),
                    "camera_position"   : e.get("camera_position", "front"),
                    "camera_yaw_deg"    : e.get("camera_yaw_deg", 0),
                    "exposure_used"     : e.get("exposure_used", 1.0),
                })
    import pandas as pd
    df = pd.DataFrame(rows)
    df.to_csv(CSV_PATH, index=False)
    print(f"  CSV built: {len(df):,} rows")
    return str(CSV_PATH)

# ============================================================
# DATASET
# ============================================================

METADATA_FIELDS = [
    "hdri_avg_lum", "hdri_peak_lum", "hdri_dyn_range", "hdri_strength",
    "hdri_saturation", "hdri_rotation_deg", "exposure_used", "hdri_tier",
]

class RelightDataset(Dataset):
    def __init__(self, csv_path, sh_npy_path, sh_json_path,
                 image_size=256, split="train", val_fraction=0.02, seed=42):
        import pandas as pd
        self.image_size = image_size

        df = pd.read_csv(csv_path)
        if "hdri_name_key" not in df.columns:
            df["hdri_name_key"] = df["hdri_name"].astype(str).apply(_normalise_hdri_key)
        if "camera_yaw_deg" not in df.columns:
            df["camera_yaw_deg"] = 0
        df["_yaw_key"] = df["camera_yaw_deg"].round(1)

        self.sh_coeffs = np.load(sh_npy_path)
        with open(sh_json_path) as f:
            self.sh_index = {_normalise_hdri_key(k): v for k, v in json.load(f).items()}

        self.meta_mean = {}; self.meta_std = {}
        for field in METADATA_FIELDS:
            if field in df.columns:
                vals = df[field].astype(float)
                self.meta_mean[field] = float(vals.mean())
                self.meta_std[field]  = float(vals.std()) + 1e-8

        self.angle_groups = {}
        for sid, grp in df.groupby("subject_id"):
            valid = grp[grp["hdri_name_key"].apply(lambda k: k in self.sh_index)]
            if len(valid) > 0:
                self.angle_groups[(int(sid), 0.0)] = valid.reset_index(drop=True)
        
        all_indices = [
            (key, i)
            for key in sorted(self.angle_groups.keys())
            for i in range(len(self.angle_groups[key]))
        ]

        indices_by_sid = {}
        for item in all_indices:
            sid = item[0][0]
            indices_by_sid.setdefault(sid, []).append(item)

        rng = random.Random(seed)
        val_indices   = []
        train_indices = []
        for sid, sid_items in sorted(indices_by_sid.items()):
            rng.shuffle(sid_items)
            n_val = max(1, int(len(sid_items) * val_fraction))
            n_val = min(n_val, len(sid_items) - 1)  
            val_indices.extend(sid_items[:n_val])
            train_indices.extend(sid_items[n_val:])

        self.indices = train_indices if split == "train" else val_indices
        self.sample_sids = [item[0][0] for item in self.indices]

        sid_counts = Counter(self.sample_sids)
        print(f"  [Dataset/{split}] {len(self.indices):,} samples  "
              f"S1={sid_counts.get(1,0):,}  S2={sid_counts.get(2,0):,}  "
              f"S3={sid_counts.get(3,0):,}  (across {len(self.angle_groups)} angle groups)")

        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])

    def _get_sh(self, key):
        norm_key = _normalise_hdri_key(str(key))
        idx      = self.sh_index.get(norm_key)
        return np.zeros(27, dtype=np.float32) if idx is None \
               else np.clip(self.sh_coeffs[idx], -10.0, 10.0)

    def _get_meta(self, row):
        import pandas as pd
        vals = []
        for f in METADATA_FIELDS:
            if f not in row.index:
                vals.append(0.0); continue
            if f == "hdri_rotation_deg":
                raw_deg = float(row[f]) if not pd.isna(row[f]) else 0.0
                vals += [math.sin(math.radians(raw_deg)),
                         math.cos(math.radians(raw_deg))]
            else:
                v = (float(row[f]) - self.meta_mean.get(f, 0)) / self.meta_std.get(f, 1)
                vals.append(float(np.clip(v, -5.0, 5.0)))
        return np.array(vals, dtype=np.float32)

    def _load_img(self, path):
        try:
            img = self.transform(Image.open(path).convert("RGB"))
            return img if not (torch.isnan(img).any() or torch.isinf(img).any()) \
                       else torch.zeros(3, self.image_size, self.image_size)
        except Exception:
            return torch.zeros(3, self.image_size, self.image_size)

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        (sid, yaw), row_i = self.indices[idx]
        grp  = self.angle_groups[(sid, yaw)]
        row1 = grp.iloc[row_i]

        if len(grp) > 1:
            cands = grp[grp["hdri_rank"] != row1["hdri_rank"]]
            if len(cands) == 0:
                cands = grp.drop(row_i)
            row2 = cands.sample(1).iloc[0] if len(cands) > 0 else row1
        else:
            row2 = row1

        img1 = self._load_img(row1["image_path"])
        img2 = self._load_img(row2["image_path"])

        sh2      = self._get_sh(row2["hdri_name_key"])
        meta2    = self._get_meta(row2)
        ze_input = torch.from_numpy(np.concatenate([sh2, meta2])).clamp(-10.0, 10.0)

        return (img1, img2, ze_input, torch.tensor(sid, dtype=torch.long),
                str(row1["image_path"]), str(row2["image_path"]))

def make_weighted_sampler(dataset):
    sid_counts = Counter(dataset.sample_sids)
    weights = [1.0 / sid_counts[s] for s in dataset.sample_sids]
    return WeightedRandomSampler(
        weights     = weights,
        num_samples = len(weights),
        replacement = True,
    )

def get_val_batch_balanced(val_ds, n_per_subject=4):
    indices_by_sid = {1: [], 2: [], 3: []}
    for i, (key, _) in enumerate(val_ds.indices):
        sid = key[0]
        if sid in indices_by_sid:
            indices_by_sid[sid].append(i)

    selected = []
    for sid in [1, 2, 3]:
        pool = indices_by_sid[sid]
        if len(pool) == 0:
            print(f"  [WARN] Subject {sid} has 0 val samples — check stratified split!")
            continue
        k = min(n_per_subject, len(pool))
        selected.extend(random.sample(pool, k))

    items   = [val_ds[i] for i in selected]
    img1s   = torch.stack([it[0] for it in items])
    img2s   = torch.stack([it[1] for it in items])
    ze_inps = torch.stack([it[2] for it in items])
    sids    = torch.stack([it[3] for it in items])

    sid_labels = [f"S{it[3].item()}" for it in items]
    print(f"  [VAL BATCH] {', '.join(sid_labels)}")
    return img1s, img2s, ze_inps, sids

# ============================================================
# MODELS
# ============================================================

class AdaGN(nn.Module):
    def __init__(self, num_channels, ze_dim=256, num_groups=32):
        super().__init__()
        while num_channels % num_groups != 0 and num_groups > 1:
            num_groups //= 2
        self.norm = nn.GroupNorm(num_groups, num_channels, affine=False)
        self.proj = nn.Linear(ze_dim, num_channels * 2)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)

    def forward(self, x, ze):
        h    = self.norm(x)
        s, b = self.proj(ze).chunk(2, dim=-1)
        return h * (s[:, :, None, None] + 1) + b[:, :, None, None]


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, ze_dim=256):
        super().__init__()
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.adagn1 = AdaGN(out_ch, ze_dim)
        self.adagn2 = AdaGN(out_ch, ze_dim)
        self.act    = nn.SiLU()
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond):
        h = self.act(self.adagn1(self.conv1(x), cond))
        h = self.act(self.adagn2(self.conv2(h), cond))
        return h + self.skip(x)

class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x): return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode="nearest"))

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        device = t.device; half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([emb.sin(), emb.cos()], dim=-1)

class TimestepEmbedding(nn.Module):
    def __init__(self, ze_dim):
        super().__init__()
        self.sinpos = SinusoidalPosEmb(ze_dim)
        self.mlp    = nn.Sequential(
            nn.Linear(ze_dim, ze_dim * 4), nn.SiLU(),
            nn.Linear(ze_dim * 4, ze_dim),
        )
    def forward(self, t): return self.mlp(self.sinpos(t))

class SelfAttention(nn.Module):
    def __init__(self, ch, num_heads=4):
        super().__init__()
        while ch % num_heads != 0 and num_heads > 1:
            num_heads //= 2
        self.norm = nn.GroupNorm(32 if ch >= 32 else ch, ch)
        self.attn = nn.MultiheadAttention(ch, num_heads, batch_first=True)
        nn.init.zeros_(self.attn.out_proj.weight)
        nn.init.zeros_(self.attn.out_proj.bias)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).view(B, C, H * W).permute(0, 2, 1)
        h, _ = self.attn(h, h, h)
        h = h.permute(0, 2, 1).view(B, C, H, W)
        return x + h

class UNet(nn.Module):
    def __init__(self, in_channels=8, base_ch=160, ch_mult=(1, 2, 2, 2), ze_dim=256):
        super().__init__()
        self.t_emb   = TimestepEmbedding(ze_dim)
        chs          = [base_ch * m for m in ch_mult]
        self.in_proj = nn.Conv2d(in_channels, chs[0], 3, padding=1)

        self.enc_blocks  = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        self.skip_chs    = []
        prev = chs[0]
        for ch in chs[:-1]:
            self.enc_blocks.append(ResBlock(prev, ch, ze_dim))
            self.skip_chs.append(ch)
            self.downsamples.append(Downsample(ch))
            prev = ch

        self.mid1     = ResBlock(prev, chs[-1], ze_dim)
        self.mid_attn = SelfAttention(chs[-1], num_heads=4)
        self.mid2     = ResBlock(chs[-1], chs[-1], ze_dim)
        prev = chs[-1]

        self.upsamples  = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        for ch in reversed(chs[:-1]):
            self.upsamples.append(Upsample(prev))
            self.dec_blocks.append(ResBlock(prev + ch, ch, ze_dim))
            prev = ch

        self.out_norm = nn.GroupNorm(32, prev)
        self.out_proj = nn.Conv2d(prev, 4, 3, padding=1)
        nn.init.zeros_(self.out_proj.weight)
        nn.init.zeros_(self.out_proj.bias)

    def forward(self, x, t, ze):
        cond  = ze + self.t_emb(t)
        h     = self.in_proj(x)
        skips = []
        for blk, down in zip(self.enc_blocks, self.downsamples):
            h = blk(h, cond); skips.append(h); h = down(h)
        h = self.mid1(h, cond)
        h = self.mid_attn(h)
        h = self.mid2(h, cond)
        for up, blk, sk in zip(self.upsamples, self.dec_blocks, reversed(skips)):
            h = up(h); h = torch.cat([h, sk], dim=1); h = blk(h, cond)
        return self.out_proj(F.silu(self.out_norm(h)))

class SidEmbed(nn.Module):
    def __init__(self, num_subjects=3, embed_dim=256):
        super().__init__()
        self.embed = nn.Embedding(num_subjects + 1, embed_dim)
        nn.init.normal_(self.embed.weight, std=0.02)

    def forward(self, sid):
        return self.embed(sid)

class LP3(nn.Module):
    def __init__(self, in_dim=36, hidden_dim=512, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, out_dim),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x): return self.net(x)

class DDPM:
    def __init__(self, T=1000, device="cuda"):
        self.T = T; self.device = device
        t_vals    = torch.linspace(0, T, T + 1, device=device)
        f         = torch.cos(((t_vals / T) + 0.008) / 1.008 * math.pi / 2) ** 2
        f         = f / f[0]
        alpha_bar = (f[1:] / f[:-1]).cumprod(0).clamp(1e-5, 1 - 1e-5)
        abp       = torch.cat([torch.ones(1, device=device), alpha_bar[:-1]])
        betas     = (1 - alpha_bar / abp).clamp(0, 0.999)
        self.betas          = betas
        self.alpha_bar      = alpha_bar
        self.sqrt_ab        = alpha_bar.sqrt()
        self.sqrt_one_m_ab  = (1 - alpha_bar).sqrt()
        self.alpha_bar_prev = abp
        self.posterior_var  = (betas * (1 - abp) / (1 - alpha_bar)).clamp(1e-20)

    def q_sample(self, x0, t, noise=None):
        if noise is None: noise = torch.randn_like(x0)
        s1 = self.sqrt_ab[t][:, None, None, None]
        s2 = self.sqrt_one_m_ab[t][:, None, None, None]
        return s1 * x0 + s2 * noise, noise

    def sample_t(self, B):
        return torch.randint(0, self.T, (B,), device=self.device)

    def sample_low_t(self, B, max_t=250):
        return torch.randint(0, max_t, (B,), device=self.device)

def load_vae(device):
    from diffusers import AutoencoderKL
    local_vae = Path("/kaggle/input/sd-vae-ft-mse")
    if local_vae.exists():
        vae = AutoencoderKL.from_pretrained(str(local_vae))
    else:
        vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
    vae.eval()
    for p in vae.parameters(): p.requires_grad_(False)
    return vae.to(device)

@torch.no_grad()
def vae_encode(vae, x): return vae.encode(x).latent_dist.sample() * 0.18215

@torch.no_grad()
def vae_decode(vae, z): return vae.decode(z / 0.18215).sample.clamp(-1, 1)

def load_lpips(device):
    from lpips import LPIPS
    fn = LPIPS(net="vgg").to(device)
    for p in fn.parameters(): p.requires_grad_(False)
    return fn

@torch.no_grad()
def ddim_sample(unet, lp3, sid_embed, ddpm, z1, ze_inp, sid_batch, n_steps, device):
    ze_hdri = lp3(ze_inp)
    ze_sid  = sid_embed(sid_batch.to(device))
    ze      = ze_hdri + ze_sid

    t_seq = torch.linspace(ddpm.T - 1, 0, n_steps, dtype=torch.long).tolist()
    B     = z1.shape[0]
    x     = torch.randn_like(z1)
    for i, t_idx in enumerate(t_seq):
        t_idx   = int(t_idx)
        t_batch = torch.full((B,), t_idx, device=device, dtype=torch.long)
        eps     = unet(torch.cat([x, z1], dim=1), t_batch, ze)
        ab      = ddpm.alpha_bar[t_idx]
        x0      = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-1.2, 1.2)
        if i < len(t_seq) - 1:
            ab_prev = ddpm.alpha_bar[int(t_seq[i + 1])]
            x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
        else:
            x = x0.clamp(-1, 1)
    return x

def color_histogram_loss(pred, target, n_bins=32):
    B, C, H, W  = pred.shape
    pred_flat   = pred.view(B, C, -1)
    target_flat = target.view(B, C, -1)
    bins  = torch.linspace(-1, 1, n_bins, device=pred.device)
    sigma = 2.0 / n_bins
    total = 0.0
    for c in range(C):
        p = pred_flat[:, c, :].unsqueeze(2)
        t = target_flat[:, c, :].unsqueeze(2)
        b = bins.view(1, 1, n_bins)
        ph = torch.exp(-((p - b)**2) / (2 * sigma**2)).mean(dim=1)
        th = torch.exp(-((t - b)**2) / (2 * sigma**2)).mean(dim=1)
        ph = ph / (ph.sum(dim=1, keepdim=True) + 1e-8)
        th = th / (th.sum(dim=1, keepdim=True) + 1e-8)
        total += F.l1_loss(ph, th)
    return total / C

def frequency_loss(pred, target):
    def sobel(x):
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]],
                          dtype=x.dtype, device=x.device).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]],
                          dtype=x.dtype, device=x.device).view(1,1,3,3)
        gray = x.mean(dim=1, keepdim=True)
        gx   = F.conv2d(gray, kx, padding=1)
        gy   = F.conv2d(gray, ky, padding=1)
        return (gx**2 + gy**2 + 1e-8).sqrt()
    return F.l1_loss(sobel(pred), sobel(target))

def saturation_loss(pred, target, fg_mask):
    fg  = fg_mask.expand_as(pred)
    loss  = torch.tensor(0.0, device=pred.device)
    count = 0
    for b in range(pred.shape[0]):
        for c in range(3):
            p_vals = pred[b, c][fg[b, c] > 0.5]
            t_vals = target[b, c][fg[b, c] > 0.5]
            if p_vals.numel() < 50: continue
            loss  = loss + (p_vals.std() - t_vals.std()).abs()
            count += 1
    return loss / max(count, 1)

def apply_fg_mask(img_tensor, subject_masks, subject_ids,
                  bg_fill=CFG["bg_fill_value"]):
    out = img_tensor.clone()
    for b_i, sid in enumerate(subject_ids.tolist()):
        sid = int(sid)
        if sid in subject_masks:
            m = subject_masks[sid]
            fg = m
            bg = 1.0 - fg
            out[b_i : b_i + 1] = img_tensor[b_i : b_i + 1] * fg + bg_fill * bg
    return out

def _t2pil(t):
    t = t.squeeze(0).clamp(-1, 1)
    return Image.fromarray(((t + 1) / 2 * 255).byte().permute(1, 2, 0).cpu().numpy())

def _np2pil(arr):
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def _err_map(pred, gt, amplify=4.0):
    diff = (pred.clamp(-1,1) - gt.clamp(-1,1)).abs() * amplify
    return _np2pil((diff.clamp(0,1).permute(1,2,0).cpu().numpy()*255).astype(np.uint8))

def _weighted_err_map(pred, gt, shadow_map, size=256, amplify=4.0):
    diff   = (pred.clamp(-1,1) - gt.clamp(-1,1)).abs()
    sm     = shadow_map.cpu().numpy() if torch.is_tensor(shadow_map) else shadow_map
    sm_t   = torch.from_numpy(sm).unsqueeze(0).unsqueeze(0)
    sm_up  = F.interpolate(sm_t, size=(pred.shape[-2], pred.shape[-1]),
                           mode="bilinear", align_corners=False).squeeze()
    weight = (1.0 - sm_up).to(pred.device)
    wtd    = (diff * weight.unsqueeze(0) * amplify).clamp(0,1)
    return _np2pil((wtd.permute(1,2,0).cpu().numpy()*255).astype(np.uint8))

def _label_panel(img, text, col=(220,220,220)):
    W, H   = img.size
    canvas = Image.new("RGB", (W, H+20), (10,10,18))
    canvas.paste(img, (0,20))
    try:
        fnt = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 10)
    except Exception:
        fnt = ImageFont.load_default()
    ImageDraw.Draw(canvas).text((3,3), text, fill=col, font=fnt)
    return canvas

def tensor_to_pil(t):
    t = t.squeeze(0).clamp(-1, 1)
    return Image.fromarray(((t+1)/2*255).byte().permute(1,2,0).cpu().numpy())

@torch.no_grad()
def save_sample_grid(vae, unet, lp3, sid_embed, ddpm,
                     val_batch, step, cfg, subject_masks):
    device = cfg["device"]
    n_show = min(12, val_batch[0].shape[0])
    img1   = val_batch[0][:n_show].to(device)
    img2   = val_batch[1][:n_show].to(device)
    ze_inp = val_batch[2][:n_show].to(device)
    sids   = val_batch[3][:n_show]

    img1_clean = apply_fg_mask(img1, subject_masks, sids)
    img2_clean = apply_fg_mask(img2, subject_masks, sids)

    unet.eval(); lp3.eval(); sid_embed.eval()
    z1        = vae_encode(vae, img1_clean)
    x_latent  = ddim_sample(unet, lp3, sid_embed, ddpm, z1, ze_inp,
                             sids, n_steps=cfg["ddim_steps"], device=device)
    pred_imgs = vae_decode(vae, x_latent)
    unet.train(); lp3.train(); sid_embed.train()

    H = W = cfg["image_size"]
    LABEL_W = 60
    grid = Image.new("RGB",
                     (LABEL_W + W*3 + 20, H*n_show + (n_show+1)*5 + 30),
                     (20, 20, 20))
    draw = ImageDraw.Draw(grid)
    try:
        fnt = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 11)
    except Exception:
        fnt = ImageFont.load_default()
    for col, label in enumerate(["Input (masked)", "GT (masked)", "Predicted"]):
        draw.text((LABEL_W + col*(W+6)+6, 5), label, fill=(200,200,200), font=fnt)
    for row in range(n_show):
        y_off   = 30 + row * (H + 5)
        sid_val = sids[row].item()
        draw.text((2, y_off + H//2 - 6), f"S{sid_val}", fill=(255,200,80), font=fnt)
        for col, t in enumerate([img1_clean[row:row+1],
                                  img2_clean[row:row+1],
                                  pred_imgs[row:row+1]]):
            grid.paste(tensor_to_pil(t), (LABEL_W + col*(W+6), y_off))

    out_path = SAMPLE_DIR / f"sample_step{step:07d}.png"
    grid.save(out_path)
    wandb_log_image("sample_grid", grid, step)
    print(f"\n  SAMPLE GRID → {out_path.name}  (step {step})\n")
    return out_path

@torch.no_grad()
def save_diagnostic_grid(vae, unet, lp3, sid_embed, ddpm, lpips_fn,
                          val_batch, step, cfg, subject_masks):
    device = cfg["device"]
    n_show = min(4, val_batch[0].shape[0])
    S      = cfg["image_size"]

    img1   = val_batch[0][:n_show].to(device)
    img2   = val_batch[1][:n_show].to(device)
    ze_inp = val_batch[2][:n_show].to(device)
    sids   = val_batch[3][:n_show]

    img1_m = apply_fg_mask(img1, subject_masks, sids)

    unet.eval(); lp3.eval(); sid_embed.eval()
    z1    = vae_encode(vae, img1_m)
    x_lat = ddim_sample(unet, lp3, sid_embed, ddpm, z1, ze_inp,
                        sids, n_steps=cfg["ddim_steps"], device=device)
    pred  = vae_decode(vae, x_lat)
    unet.train(); lp3.train(); sid_embed.train()

    N_COLS = 5; PAD = 4; HDR = 50; PH = S + 20
    GW = N_COLS * S + (N_COLS + 1) * PAD
    GH = HDR + n_show * (PH + PAD) + PAD
    grid = Image.new("RGB", (GW, GH), (8, 8, 14))
    draw = ImageDraw.Draw(grid)
    try:
        fh = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 13)
        fs = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 10)
    except Exception:
        fh = fs = ImageFont.load_default()

    draw.text((PAD, 6),
              f"RELIGHT v15  step={step:,}  [160 Channels]",
              fill=(255,210,50), font=fh)
    col_labels = [
        ("Input (masked)",  (160,160,160)),
        ("Ground Truth",    (80, 200, 80)),
        ("Prediction",      (80, 150, 255)),
        ("Error ×4",        (200,200, 80)),
        ("Shadow-wtd Err",  (255,120, 40)),
    ]
    for ci, (label, col) in enumerate(col_labels):
        draw.text((PAD + ci*(S+PAD)+2, HDR-18), label, fill=col, font=fs)

    for ri in range(n_show):
        src_t  = img1_m[ri]
        gt_t   = img2[ri]
        pred_t = pred[ri]

        gt_lum      = (0.2126*gt_t[0] + 0.7152*gt_t[1] + 0.0722*gt_t[2]).cpu()
        gt_lum_norm = (gt_lum - gt_lum.min()) / (gt_lum.max() - gt_lum.min() + 1e-8)

        panels = [
            _label_panel(_t2pil(src_t), f"Input S{sids[ri].item()}"),
            _label_panel(_t2pil(gt_t), "GT"),
            _label_panel(_t2pil(pred_t), "Pred"),
            _label_panel(_err_map(pred_t, gt_t, 4.0), "Error ×4"),
            _label_panel(_weighted_err_map(pred_t, gt_t,
                         gt_lum_norm.numpy(), S), "Shadow-wtd"),
        ]
        y_off = HDR + ri * (PH + PAD) + PAD
        for ci, panel in enumerate(panels):
            x = PAD + ci * (S + PAD)
            grid.paste(panel.convert("RGB").resize((S, PH), Image.LANCZOS), (x, y_off))

    out_path = DIAG_DIR / f"diag_step{step:07d}.png"
    grid.save(str(out_path))
    print(f"\n  [DIAG] Saved → {out_path.name}\n")
    wandb_log_image("diag_grid", grid, step)
    return out_path

@torch.no_grad()
def run_val_loop(vae, unet, lp3, sid_embed, ddpm,
                 val_loader, lpips_fn, cfg, step, subject_masks):
    device = cfg["device"]
    unet.eval(); lp3.eval(); sid_embed.eval()
    total_mse = 0.0; total_lpips = 0.0; n_batches = 0

    for batch in val_loader:
        img1, img2, ze_inp, sids, _, _ = batch
        img1   = img1.to(device)
        img2   = img2.to(device)
        ze_inp = ze_inp.to(device)
        img1_m = apply_fg_mask(img1, subject_masks, sids)
        img2_m = apply_fg_mask(img2, subject_masks, sids)
        z1     = vae_encode(vae, img1_m)
        x_lat  = ddim_sample(unet, lp3, sid_embed, ddpm, z1, ze_inp,
                             sids, n_steps=20, device=device)
        pred   = vae_decode(vae, x_lat)
        total_mse   += F.mse_loss(pred, img2_m).item()
        total_lpips += lpips_fn(pred, img2_m).mean().item()
        n_batches   += 1
        if n_batches >= 20: break

    unet.train(); lp3.train(); sid_embed.train()
    avg_mse   = total_mse   / max(n_batches, 1)
    avg_lpips = total_lpips / max(n_batches, 1)
    print(f"  [VAL] step={step}  mse={avg_mse:.4f}  lpips={avg_lpips:.4f}")
    wandb_log({"val/mse": avg_mse, "val/lpips": avg_lpips, "step": step})
    return avg_mse, avg_lpips

# ============================================================
# MAIN TRAINING FUNCTION
# ============================================================

def train():
    init_wandb()
    torch.backends.cudnn.benchmark = True

    print("\n--- Step 1: SH Precomputation ---")
    sh_npy, sh_json = precompute_sh_if_needed()

    print("\n--- Step 2: Build CSV ---")
    csv_path = build_csv_if_needed()

    print("\n--- Step 3: Datasets ---")
    train_ds = RelightDataset(csv_path, sh_npy, sh_json,
                              image_size=CFG["image_size"], split="train",
                              val_fraction=CFG["val_fraction"])
    val_ds   = RelightDataset(csv_path, sh_npy, sh_json,
                              image_size=CFG["image_size"], split="val",
                              val_fraction=CFG["val_fraction"])

    sample_ze = train_ds[0][2]
    assert sample_ze.shape[0] == 36, (
        f"ze_input dim is {sample_ze.shape[0]}, expected 36. "
        f"Check METADATA_FIELDS and _get_meta()."
    )
    print(f"  [CHECK] ze_input dim = {sample_ze.shape[0]} ✓ (27 SH + 9 meta = 36)")

    sampler = make_weighted_sampler(train_ds)
    nw      = CFG["num_workers"]
    pf      = 2 if nw > 0 else None
    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                              sampler=sampler,
                              num_workers=nw, pin_memory=True,
                              persistent_workers=nw > 0,
                              prefetch_factor=pf,
                              drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                              num_workers=nw, pin_memory=True,
                              persistent_workers=nw > 0,
                              prefetch_factor=pf)
    print(f"  Train: {len(train_ds):,}  Val: {len(val_ds):,}")

    print("\n--- Step 4: Models ---")
    device    = CFG["device"]
    vae       = load_vae(device)
    lp3       = LP3(in_dim=36, hidden_dim=512, out_dim=CFG["ze_dim"]).to(device)
    sid_embed = SidEmbed(num_subjects=NUM_SUBJECTS,
                         embed_dim=SID_EMBED_DIM).to(device)
    unet      = UNet(in_channels=8, base_ch=CFG["base_ch"],
                     ze_dim=CFG["ze_dim"]).to(device)
    ddpm      = DDPM(T=CFG["T"], device=device)
    lpips_fn  = load_lpips(device)
    scaler    = GradScaler("cuda", enabled=CFG["use_amp"])

    n_params = sum(p.numel() for p in
                   list(lp3.parameters()) +
                   list(sid_embed.parameters()) +
                   list(unet.parameters()))
    print(f"  Params: {n_params/1e6:.1f}M  "
          f"(+{sum(p.numel() for p in sid_embed.parameters())/1e3:.0f}K for SidEmbed)")

    # ── Masks ─────────────────────────────────────────────────
    subject_masks = {}
    _mask_search_dirs = [
        KAGGLE_DATA_ROOT / "datasets/devsachan/relight-masks",
        KAGGLE_DATA_ROOT / "datasets/coding12341234/relight-masks",
        KAGGLE_DATA_ROOT / "relight-masks",
    ]
    _mask_dir = None
    for _d in _mask_search_dirs:
        if _d.exists():
            _mask_dir = _d; break
    if _mask_dir is None:
        _found = list(KAGGLE_DATA_ROOT.rglob("mask_subject1.npy"))
        if _found: _mask_dir = _found[0].parent

    if _mask_dir is not None:
        print(f"  Loading masks from: {_mask_dir}")
        for sid in [1, 2, 3]:
            _mp = _mask_dir / f"mask_subject{sid}.npy"
            if _mp.exists():
                arr    = np.load(_mp).astype(np.float32)
                mask_t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(device)
                subject_masks[sid] = mask_t
                print(f"    Subject {sid}: ✓  fg={arr.mean():.2%}  shape={arr.shape}")
            else:
                print(f"    Subject {sid}: NOT FOUND — no masking for S{sid}")
        print(f"  Masks ready: {len(subject_masks)}/3 subjects")
    else:
        raise FileNotFoundError(
            "Could not find relight-masks dataset. "
            "Upload mask_subject1.npy, mask_subject2.npy, mask_subject3.npy."
        )

    latent_array  = None
    latent_index  = {}

    # ── Resume / fresh start — LOAD WEIGHTS FIRST (before compile) ─────
    step      = 0
    best_loss = float("inf")
    nan_count = 0
    ckpt      = None   

    _pre_unet_norm = unet.in_proj.weight.detach().float().norm().item()
    _pre_lp3_norm  = lp3.net[0].weight.detach().float().norm().item()

    if _dst_ckpt.exists() and not FORCE_FRESH_START:
        ckpt = torch.load(_dst_ckpt, map_location=device)

        def _strip(sd):
            return {k.replace("_orig_mod.", ""): v for k, v in sd.items()}

        unet_sd = _strip(ckpt["unet"])
        lp3_sd  = _strip(ckpt["lp3"])

        t_unet = transfer_weights_with_resize(unet, unet_sd)
        print(f"  [RESUME] UNet : transferred {t_unet}/{len(unet_sd)} layers with resizing")
        
        missing_l, unexpected_l = lp3.load_state_dict(lp3_sd, strict=False)
        print(f"  [RESUME] LP3  : loaded={len(lp3_sd)-len(unexpected_l)} "
              f"missing={len(missing_l)} unexpected={len(unexpected_l)}")

        if "sid_embed" in ckpt:
            sid_sd = _strip(ckpt["sid_embed"])
            missing_s, unexpected_s = sid_embed.load_state_dict(sid_sd, strict=False)
            print(f"  [RESUME] SidEmb: loaded={len(sid_sd)-len(unexpected_s)} "
                  f"missing={len(missing_s)} unexpected={len(unexpected_s)}")
        else:
            print("  [RESUME] No SidEmbed in checkpoint — random init")

        step      = ckpt["step"]
        best_loss = ckpt.get("best_loss", float("inf"))

        _post_unet_norm = unet.in_proj.weight.detach().float().norm().item()
        _post_lp3_norm  = lp3.net[0].weight.detach().float().norm().item()
        _unet_changed = abs(_pre_unet_norm - _post_unet_norm) > 1e-4
        _lp3_changed  = abs(_pre_lp3_norm  - _post_lp3_norm)  > 1e-4
        print(f"  [SANITY] unet.in_proj norm: {_pre_unet_norm:.4f} → {_post_unet_norm:.4f}  "
              f"{'✓ CHANGED' if _unet_changed else '✗✗ UNCHANGED — LOAD FAILED!'}")
        print(f"  [SANITY] lp3[0]     norm : {_pre_lp3_norm:.4f} → {_post_lp3_norm:.4f}  "
              f"{'✓ CHANGED' if _lp3_changed else '✗✗ UNCHANGED — LOAD FAILED!'}")
        if not (_unet_changed and _lp3_changed):
            raise RuntimeError(
                "Weights did NOT change after load_state_dict."
            )

        print(f"  [RESUME] ✓  step={step:,}  best_loss={best_loss:.4f}")

    if hasattr(torch, "compile"):
        print("  Compiling models with torch.compile...")
        unet      = torch.compile(unet)
        lp3       = torch.compile(lp3)
        sid_embed = torch.compile(sid_embed)
        print("  torch.compile done ✓")

    params = (list(lp3.parameters()) +
              list(sid_embed.parameters()) +
              list(unet.parameters()))
    opt = torch.optim.AdamW(params, lr=CFG["lr"], weight_decay=CFG["wd"],
                             betas=(0.9, 0.999))
    
    # --- SEAMLESS LR RESUME PATCH ---
    # Extract the true decayed LR from the checkpoint to prevent resetting to 3e-5
    if ckpt is not None and "opt" in ckpt:
        try:
            last_lr = ckpt["opt"]["param_groups"][0]["lr"]
            for group in opt.param_groups:
                group["lr"] = last_lr
        except Exception:
            pass

    for group in opt.param_groups:
        group.setdefault('initial_lr', CFG['lr'])

    # --- THE LR FIX ---
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, 
        T_max=CFG["total_steps"], 
        eta_min=CFG["lr_min"],
        last_epoch=step - 1 if step > 0 else -1  
    )

    # FORCE FRESH OPTIMIZER FOR V15:
    # Parameter shapes have changed, old Adam moments will crash.
    if ckpt is not None and "opt" in ckpt and False:
        pass # Intentionally bypassed
    else:
        print("  [RESUME] Optimizer restore bypassed — using fresh optimizer for resized architecture.")

    print(f"\n--- Training from step {step} to {CFG['total_steps']} ---\n")

    accum_steps = CFG["accum_steps"]
    lpips_every = CFG["lpips_every"]
    lpips_max_t = CFG["lpips_max_t"]
    debug_end   = (step + DEBUG_STEPS) if DEBUG_MODE else CFG["total_steps"]

    loss_accum = {k: 0.0 for k in
                  ["total","mse","lpips","ch","shadow","freq","sat","color_mv"]}
    t0          = time.time()
    loader_iter = iter(train_loader)
    opt.zero_grad(set_to_none=True)

    pbar = tqdm(total=debug_end if DEBUG_MODE else CFG["total_steps"],
                initial=step,
                desc="Training [DEBUG]" if DEBUG_MODE else "Training",
                unit="step", dynamic_ncols=False, ncols=130, mininterval=10.0)

    while step < debug_end:

        acc       = {k: 0.0 for k in loss_accum}
        do_lpips  = (step % lpips_every == 0)
        skip_step = False

        for accum_i in range(accum_steps):
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(train_loader)
                batch       = next(loader_iter)

            img1, img2, ze_input, sid_batch, paths1, paths2 = batch
            img1      = img1.to(device)
            img2      = img2.to(device)
            ze_input  = ze_input.to(device)

            with autocast("cuda", enabled=CFG["use_amp"]):

                img1_clean = apply_fg_mask(img1, subject_masks, sid_batch)
                img2_clean = apply_fg_mask(img2, subject_masks, sid_batch)

                z1 = vae_encode(vae, img1_clean)
                z2 = vae_encode(vae, img2_clean)

                ze_hdri = lp3(ze_input)
                ze_sid  = sid_embed(sid_batch.to(device))
                ze      = ze_hdri + ze_sid

                if torch.isnan(ze).any() or torch.isinf(ze).any():
                    print(f"  [WARN] NaN/Inf in ze at step {step} — skipping")
                    nan_count += 1; skip_step = True; break

                t_batch  = ddpm.sample_t(img1.shape[0])
                xt, eps  = ddpm.q_sample(z2, t_batch)
                eps_pred = unet(torch.cat([xt, z1], dim=1), t_batch, ze)

                fg_px      = torch.cat([
                    subject_masks.get(int(s),
                        torch.ones(1, 1, CFG["image_size"], CFG["image_size"],
                                   device=device))
                    for s in sid_batch
                ], dim=0)
                fg_lat     = F.interpolate(fg_px, size=(32,32),
                                           mode="bilinear", align_corners=False)
                weight_map = fg_lat * CFG["fg_weight"] + (1 - fg_lat) * CFG["bg_weight"]
                mse        = (weight_map * (eps_pred - eps).abs()).mean()

                with torch.no_grad():
                    gt_lum = (0.2126 * img2_clean[:,0]
                            + 0.7152 * img2_clean[:,1]
                            + 0.0722 * img2_clean[:,2])
                    fg_mask_2d = fg_px.squeeze(1)
                    _lum_inf   = gt_lum.masked_fill(fg_mask_2d < 0.5,  float("inf"))
                    _lum_ninf  = gt_lum.masked_fill(fg_mask_2d < 0.5, -float("inf"))
                    gt_min = _lum_inf.flatten(1).min(dim=1)[0][:,None,None]
                    gt_max = _lum_ninf.flatten(1).max(dim=1)[0][:,None,None]
                    gt_shadow_map = ((gt_lum - gt_min) /
                                     (gt_max - gt_min + 1e-8)).clamp(0,1) * fg_mask_2d

                lpips_val   = torch.tensor(0.0, device=device)
                ch_val      = torch.tensor(0.0, device=device)
                shadow_loss = torch.tensor(0.0, device=device)
                freq_val    = torch.tensor(0.0, device=device)
                sat_val     = torch.tensor(0.0, device=device)
                color_mv    = torch.tensor(0.0, device=device)

                if do_lpips and accum_i == 0:
                    try:
                        t_lp       = ddpm.sample_low_t(img1.shape[0], max_t=lpips_max_t)
                        xt_l, _    = ddpm.q_sample(z2, t_lp)
                        eps_pred_l = unet(torch.cat([xt_l, z1], dim=1), t_lp, ze)
                        ab         = ddpm.sqrt_ab[t_lp][:, None, None, None]
                        oab        = ddpm.sqrt_one_m_ab[t_lp][:, None, None, None]
                        z2p        = (xt_l - oab * eps_pred_l) / ab
                        img2p      = vae_decode(vae, z2p)

                        if not (torch.isnan(img2p).any() or torch.isinf(img2p).any()):

                            lpips_val = lpips_fn(img2p, img2_clean).mean()
                            ch_val    = color_histogram_loss(img2p, img2_clean)
                            sat_val   = saturation_loss(img2p, img2_clean, fg_px)

                            fg_up       = fg_mask_2d.unsqueeze(1)
                            sw_fg       = ((1.0 - gt_shadow_map)**6).unsqueeze(1) * fg_up
                            shadow_loss = (sw_fg * (img2p - img2_clean).abs()).mean()

                            pred_lum   = (0.2126*img2p[:,0]
                                        + 0.7152*img2p[:,1]
                                        + 0.0722*img2p[:,2])
                            _p_inf     = pred_lum.masked_fill(fg_mask_2d < 0.5,  float("inf"))
                            _p_ninf    = pred_lum.masked_fill(fg_mask_2d < 0.5, -float("inf"))
                            pred_min   = _p_inf.flatten(1).min(dim=1)[0][:,None,None]
                            pred_max   = _p_ninf.flatten(1).max(dim=1)[0][:,None,None]
                            pred_sm    = ((pred_lum - pred_min) /
                                          (pred_max - pred_min + 1e-8)).clamp(0,1) * fg_mask_2d
                            
                            # --- THE SAFE-PACE PIVOT ---
                            # (diff + 1.5 * diff**2)
                            # The 'diff' keeps it stable for fine lines.
                            # The '1.5 * diff**2' forces the grey 'smudges' to go black fast.
                            sm_diff = (pred_sm - gt_shadow_map).abs()
                            shadow_error_sting = (sm_diff + 1.5 * (sm_diff**2)).mean()
                            
                            shadow_loss = shadow_loss + shadow_error_sting

                            freq_val = frequency_loss(img2p, img2_clean)

                            fg_lat_mask = F.interpolate(fg_px, size=(32,32),
                                                        mode="bilinear").squeeze(1)
                            for ch in range(4):
                                p_ch = z2p[:, ch][fg_lat_mask > 0.5]
                                t_ch = z2[:,  ch][fg_lat_mask > 0.5]
                                if p_ch.numel() > 50:
                                    color_mv = color_mv + (
                                        (p_ch.mean() - t_ch.mean()).abs() +
                                        (p_ch.var()  - t_ch.var() ).abs()
                                    )
                            color_mv = color_mv / 4.0

                    except Exception as e:
                        print(f"  [WARN] perceptual losses failed at step {step}: {e}")

                loss = (
                    CFG["mse_weight"]       * mse
                    + CFG["lpips_weight"]     * lpips_val
                    + CFG["ch_loss_weight"]   * ch_val
                    + CFG["shadow_weight"]    * shadow_loss
                    + CFG["freq_loss_weight"] * freq_val
                    + CFG["sat_loss_weight"]  * sat_val
                    + CFG["color_mv_weight"]  * color_mv
                ) / accum_steps

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"  [WARN] NaN/Inf loss at step {step} — skipping")
                nan_count += 1; skip_step = True; break

            scaler.scale(loss).backward()

            acc["total"]    += loss.item()
            acc["mse"]      += mse.item()        / accum_steps
            acc["lpips"]    += lpips_val.item()  / accum_steps
            acc["ch"]       += ch_val.item()     / accum_steps
            acc["shadow"]   += shadow_loss.item()/ accum_steps
            acc["freq"]     += freq_val.item()   / accum_steps
            acc["sat"]      += sat_val.item()    / accum_steps
            acc["color_mv"] += color_mv.item()   / accum_steps

        if skip_step:
            opt.zero_grad(set_to_none=True)
            if accum_i > 0:
                try: scaler.update()
                except Exception: pass
            step += 1; pbar.update(1)
            continue

        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); scheduler.step()
        opt.zero_grad(set_to_none=True)

        for k in loss_accum: loss_accum[k] += acc[k]
        step += 1; pbar.update(1)

        log_every = 10 if DEBUG_MODE else CFG["log_every"]
        if step % log_every == 0:
            n       = log_every
            lr      = scheduler.get_last_lr()[0]
            elapsed = time.time() - t0
            its     = step / elapsed
            eta_h   = (CFG["total_steps"] - step) / its / 3600
            al   = loss_accum["total"]  / n
            am   = loss_accum["mse"]    / n
            ap   = loss_accum["lpips"]  / n
            ash  = loss_accum["shadow"] / n
            asat = loss_accum["sat"]    / n
            print(
                f"[{step:7d}/{CFG['total_steps']}]  "
                f"loss={al:.4f}  mse={am:.4f}  lpips={ap:.4f}  "
                f"shadow={ash:.4f}  sat={asat:.4f}  "
                f"lr={lr:.2e}  {its:.1f}it/s  eta={eta_h:.1f}h  "
                f"nan={nan_count}"
            )
            wandb_log({
                "train/loss": al, "train/mse": am, "train/lpips": ap,
                "train/shadow": ash, "train/sat": asat,
                "train/lr": lr, "perf/its": its, "perf/eta_h": eta_h,
                "train/nan_count": nan_count, "step": step,
            })
            pbar.set_postfix(loss=f"{al:.4f}", mse=f"{am:.4f}",
                             sat=f"{asat:.4f}", eta=f"{eta_h:.1f}h")
            loss_accum = {k: 0.0 for k in loss_accum}

        sample_trigger = DEBUG_SAMPLE_EVERY if DEBUG_MODE else CFG["sample_every"]
        if step % sample_trigger == 0:
            rand_batch = get_val_batch_balanced(val_ds, n_per_subject=4)
            save_sample_grid(vae, unet, lp3, sid_embed, ddpm,
                             rand_batch, step, CFG, subject_masks)

        diag_trigger = DEBUG_DIAG_AT if DEBUG_MODE else CFG["diag_every"]
        should_diag  = (DEBUG_MODE and step == debug_end - 1) or \
                       (step % diag_trigger == 0)
        if should_diag:
            rand_diag = get_val_batch_balanced(val_ds, n_per_subject=2)
            save_diagnostic_grid(vae, unet, lp3, sid_embed, ddpm, lpips_fn,
                                 rand_diag, step, CFG, subject_masks)

        if not DEBUG_MODE and step % CFG["val_every"] == 0:
            val_mse, val_lpips = run_val_loop(
                vae, unet, lp3, sid_embed, ddpm,
                val_loader, lpips_fn, CFG, step, subject_masks)
            combined = val_mse + 0.3 * val_lpips
            if combined < best_loss:
                best_loss = combined
                torch.save({
                    "step": step, "best_loss": best_loss,
                    "lp3": lp3.state_dict(),
                    "sid_embed": sid_embed.state_dict(),
                    "unet": unet.state_dict(),
                    "opt": opt.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "scaler": scaler.state_dict(),
                }, CKPT_DIR / "best.pt")
                print(f"  [BEST] step={step}  combined={combined:.4f}")

        if not DEBUG_MODE and step % CFG["save_last_every"] == 0:
            torch.save({
                "step": step, "best_loss": best_loss,
                "lp3": lp3.state_dict(),
                "sid_embed": sid_embed.state_dict(),
                "unet": unet.state_dict(),
                "opt": opt.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
            }, _dst_ckpt)
            print(f"  [LAST] last.pt saved at step {step}")

        if not DEBUG_MODE and step % CFG["save_ckpt_every"] == 0:
            ckpt_path = CKPT_DIR / f"ckpt_{step:07d}.pt"
            torch.save({
                "step": step, "best_loss": best_loss,
                "lp3": lp3.state_dict(),
                "sid_embed": sid_embed.state_dict(),
                "unet": unet.state_dict(),
                "opt": opt.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
            }, ckpt_path)
            print(f"  [CKPT] Saved {ckpt_path.name}")

            diag_files = sorted(DIAG_DIR.glob("*.png"))
            if diag_files:
                zip_path = CKPT_DIR / f"diag_grids_{step:07d}.zip"
                with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
                    for df_path in diag_files:
                        zf.write(df_path, df_path.name)
                print(f"  [DIAG] Zipped {len(diag_files)} grids → {zip_path.name}")

    pbar.close()

    if DEBUG_MODE:
        return

    torch.save({
        "step": step, "best_loss": best_loss,
        "lp3": lp3.state_dict(),
        "sid_embed": sid_embed.state_dict(),
        "unet": unet.state_dict(),
        "opt": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
    }, _dst_ckpt)
    print(f"\n  Training complete at step {step}  best_loss={best_loss:.4f}")
    wandb_finish()

# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    train()

  [RESUME] last.pt exists  (step=405000, loss=0.0617)
  [RESUME] master_metadata.csv: already exists
  Found data root: /kaggle/input/datasets/devsachaniitk/alexander/alexander
  [DEVICE] cuda
  [GPU] t4 — batch=4 base_ch=160
  [WandB] WANDB_API_KEY not found — logging disabled.

--- Step 1: SH Precomputation ---
  SH cache found — skipping.

--- Step 2: Build CSV ---
  CSV found — skipping build.

--- Step 3: Datasets ---
  [Dataset/train] 61,389 samples  S1=20,463  S2=20,463  S3=20,463  (across 3 angle groups)
  [Dataset/val] 1,251 samples  S1=417  S2=417  S3=417  (across 3 angle groups)
  [CHECK] ze_input dim = 36 ✓ (27 SH + 9 meta = 36)
  Train: 61,389  Val: 1,251

--- Step 4: Models ---


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
  Params: 22.9M  (+1K for SidEmbed)
  Loading masks from: /kaggle/input/datasets/devsachan/relight-masks
    Subject 1: ✓  fg=28.65%  shape=(256, 256)
    Subject 2: ✓  fg=34.21%  shape=(256, 256)
    Subject 3: ✓  fg=41.44%  shape=(256, 256)
  Masks ready: 3/3 subjects
  [RESUME] UNet : transferred 100/100 layers with resizing
  [RESUME] LP3  : loaded=14 missing=0 unexpected=0
  [RESUME] SidEmb: loaded=1 missing=0 unexpected=0
  [SANITY] unet.in_proj norm: 7.2964 → 7.3460  ✓ CHANGED
  [SANITY] lp3[0]     norm : 32.1228 → 37.2115  ✓ CHANGED
  [RESUME] ✓  step=405,000  best_loss=0.0617
  Compiling models with torch.compile...
  torch.compile done ✓
  [RESUME] Optimizer restore bypassed — using fresh optimizer for resized architecture.

--- Training from step 405000 to 500000 ---



...

[ 420000/500000]  loss=0.4135  mse=0.2106  lpips=0.0150  shadow=0.0052  sat=0.0017  lr=1.24e-05  33.8it/s  eta=0.7h  nan=0
  [VAL BATCH] S1, S1, S1, S1, S2, S2, S2, S2, S3, S3, S3, S3

  SAMPLE GRID → sample_step0420000.png  (step 420000)

  [VAL BATCH] S1, S1, S2, S2, S3, S3

  [DIAG] Saved → diag_step0420000.png



Training:  84%|█████████████████▋   | 420000/500000 [3:27:10<18:02:20,  1.23step/s, eta=0.7h, loss=0.4135, mse=0.2106, sat=0.0017]

  [VAL] step=420000  mse=0.0296  lpips=0.1287
  [LAST] last.pt saved at step 420000
  [CKPT] Saved ckpt_0420000.pt
  [DIAG] Zipped 2 grids → diag_grids_0420000.zip


Training:  84%|█████████████████▋   | 420014/500000 [3:27:34<30:25:39,  1.37s/step, eta=0.7h, loss=0.4135, mse=0.2106, sat=0.0017]

KeyboardInterrupt: 

In [ ]:
import glob

# Find all .pt files anywhere in input
found = glob.glob("/kaggle/input/**/*.pt", recursive=True)
print("All .pt files:")
for f in found:
    print(f"  {f}")

# Also list all input datasets
import os
print("\nInput datasets:")
for item in os.listdir("/kaggle/input"):
    print(f"  {item}")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    for f in files:
        if f.endswith(".pt"):
            print(os.path.join(root, f))

In [ ]:
import shutil
import zipfile
from pathlib import Path
from IPython.display import display, HTML

CKPT_DIR = Path("/kaggle/working/checkpoints")
OUT_DIR = Path("/kaggle/working")

# Save safety copy
dst = OUT_DIR / "last_backup.pt"
shutil.copy(CKPT_DIR / "last.pt", dst)
print("✓ Safety copy saved")

# Zip to working root
zip_path = OUT_DIR / "ckpt_35k.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in ["last.pt"]:
        p = CKPT_DIR / name
        zf.write(p, p.name)
        print(f"Added: {name} ({p.stat().st_size/1e6:.1f} MB)")

size_mb = zip_path.stat().st_size / 1e6
print(f"\n✓ Zip size: {size_mb:.1f} MB")

if size_mb > 1000:
    print("⚠️ Too large for Kaggle download")

    for name in ["last.pt"]:
        shutil.copy(CKPT_DIR / name, OUT_DIR / name)
        print(f"Copied {name} to /kaggle/working/")
else:
    display(HTML(
        '<a href="/kaggle/working/ckpt_35k.zip" download>'
        '⬇️ Download ckpt_35k.zip</a>'
    ))

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  RELIGHT v15 [160-CHANNEL] — COMPLETE INFERENCE & COMPOSITING CELL           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  PART A — Quantitative Metrics: MSE · PSNR · SSIM · LPIPS                    ║
║  PART B — Standard Eval Grids (Input | GT | Predicted with GT Background)    ║
║  PART C — Rotation Analysis Grids (1 Face, 1 HDRI, Multiple Angles)          ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ── CONFIG ────────────────────────────────────────────────────────────────────
EVAL_CFG = {
    "n_eval"           : 60,
    "ddim_steps"       : 150,
    "batch_size"       : 4,
    "samples_per_grid" : 32,
    "n_grids"          : 20,
    "image_size"       : 256,
    "grid_cols"        : 4,
    "val_fraction"     : 0.02,
    "val_seed"         : 42,
    "n_rotation_grids" : 5,    # How many Part C rotation grids to generate
}

import os, sys, json, math, random, time, shutil, warnings, subprocess, zipfile
import numpy as np, base64, io
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}  |  PyTorch: {torch.__version__}")

def _pip(pkg, import_as=None):
    try: __import__(import_as or pkg)
    except ImportError:
        print(f"  Installing {pkg}...")
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])

_pip("scikit-image","skimage"); _pip("lpips"); _pip("rembg[cpu]","rembg")

from skimage.metrics import structural_similarity as _ssim_fn
from lpips import LPIPS as _LPIPS

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT    = Path("/kaggle/input")
WORK     = Path("/kaggle/working")
EVAL_DIR = WORK / "eval_outputs"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# ── Find best checkpoint ──────────────────────────────────────────────────────
def _valid_pt(p):
    try:
        ck = torch.load(p, map_location="cpu")
        return True, ck.get("step",0), ck.get("best_loss",float("inf"))
    except: return False, 0, float("inf")

best_path, best_loss = None, float("inf")
print("\n  Scanning checkpoints...")
for ds in ["relight-best","relight-checkpoint"]:
    for root in INPUT.rglob(ds):
        if not root.is_dir(): continue
        for pt in sorted(root.glob("*.pt")):
            ok,step,loss = _valid_pt(pt)
            flag = ""
            if ok and loss < best_loss:
                best_loss=loss; best_path=pt; flag="  <- selected"
            print(f"    {pt.name:<22} {'step='+str(step)+' loss='+f'{loss:.4f}' if ok else 'CORRUPT'}{flag}")

if best_path is None:
    raise FileNotFoundError("No valid checkpoint found.")
ckpt      = torch.load(best_path, map_location=DEVICE)
ckpt_step = ckpt.get("step","?")
print(f"\n  Using: {best_path.name}  step={ckpt_step}  loss={best_loss:.4f}")

# ── HDRI cache ────────────────────────────────────────────────────────────────
HDRI_CACHE = next((p for p in INPUT.rglob("hdri_cache") if p.is_dir()), None)
if HDRI_CACHE is None: raise FileNotFoundError("hdri_cache not found")
print(f"  HDRI cache : {HDRI_CACHE}")

# ── Metadata CSV ──────────────────────────────────────────────────────────────
CSV_PATH = next((p for p in INPUT.rglob("master_metadata.csv")), None)
if CSV_PATH is None: raise FileNotFoundError("master_metadata.csv not found")
import pandas as pd
df_meta = pd.read_csv(str(CSV_PATH))
print(f"  Metadata   : {len(df_meta)} rows")

# ── SH cache ─────────────────────────────────────────────────────────────────
SH_DIR  = WORK/"sh_cache"
SH_NPY  = SH_DIR/"sh_coeffs.npy"
SH_JSON = SH_DIR/"sh_index.json"

if not SH_NPY.exists():
    for _p in INPUT.rglob("sh_coeffs.npy"):
        SH_NPY  = _p
        SH_JSON = _p.parent/"sh_index.json"
        SH_DIR  = _p.parent
        print(f"  SH cache found in dataset: {SH_DIR}")
        break

if not SH_NPY.exists():
    print("  SH cache not found — precomputing from HDRI cache (~2 min)...")
    SH_DIR.mkdir(parents=True, exist_ok=True)
    def _build_sh_dirs(H, W):
        phi   = (np.arange(W)+0.5)/W*2*np.pi
        theta = (np.arange(H)+0.5)/H*np.pi
        THETA, PHI = np.meshgrid(theta, phi, indexing='ij')
        st=np.sin(THETA); ct=np.cos(THETA); sp=np.sin(PHI); cp=np.cos(PHI)
        x=st*cp; y=st*sp; z=ct
        Y=np.stack([0.282095*np.ones_like(x), 0.488603*y,0.488603*z,0.488603*x,
                    1.092548*x*y,1.092548*y*z, 0.315392*(3*z*z-1),1.092548*x*z,
                    0.546274*(x*x-y*y)],axis=-1)
        return Y, st*(np.pi/H)*(2*np.pi/W)

    def _compute_sh(hdr_npy):
        img=((hdr_npy.astype(np.float64)+1.0)/2.0).clip(0)
        C,H,W=img.shape; Y,sa=_build_sh_dirs(H,W)
        return np.concatenate([(img[c][:,:,None]*Y*sa[:,:,None]).sum(axis=(0,1))
                                for c in range(C)]).astype(np.float32)

    hdr_files = sorted(HDRI_CACHE.glob("*_hdr.npy"))
    all_coeffs=[]; sh_idx={}
    for i,fp in enumerate(hdr_files):
        arr=np.load(fp)
        if arr.ndim!=3 or arr.shape[0]!=3: continue
        key=fp.stem.replace("_hdr","").strip().lower()
        sh_idx[key]=len(all_coeffs)
        all_coeffs.append(_compute_sh(arr))
        if (i+1)%100==0: print(f"    [{i+1}/{len(hdr_files)}]")
    arr_out=np.stack(all_coeffs)
    np.save(SH_NPY, arr_out)
    with open(SH_JSON,"w") as f: json.dump(sh_idx, f)
    print(f"  SH precomputed: {arr_out.shape} ✓")

sh_coeffs_all = np.load(SH_NPY)
with open(SH_JSON) as f: _sh_raw = json.load(f)

def _nk(k):
    k = k.strip().lower()
    for e in (".hdr",".exr",".png"): k=k[:-len(e)] if k.endswith(e) else k
    return k

sh_index = {_nk(k):v for k,v in _sh_raw.items()}

# =============================================================================
# MODEL DEFINITIONS -- v15 (160 Channels)
# =============================================================================

class AdaGN(nn.Module):
    def __init__(self,nc,ze=256,ng=32):
        super().__init__()
        while nc%ng!=0 and ng>1: ng//=2
        self.norm=nn.GroupNorm(ng,nc,affine=False)
        self.proj=nn.Linear(ze,nc*2)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)
    def forward(self,x,ze):
        h=self.norm(x); s,b=self.proj(ze).chunk(2,dim=-1)
        return h*(s[:,:,None,None]+1)+b[:,:,None,None]

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, ze_dim=256):
        super().__init__()
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.adagn1 = AdaGN(out_ch, ze_dim)
        self.adagn2 = AdaGN(out_ch, ze_dim)
        self.act    = nn.SiLU()
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, cond):
        h = self.act(self.adagn1(self.conv1(x), cond))
        h = self.act(self.adagn2(self.conv2(h), cond))
        return h + self.skip(x)

class Downsample(nn.Module):
    def __init__(self, ch): 
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x): 
        return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, ch): 
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x): 
        return self.conv(F.interpolate(x, scale_factor=2, mode="nearest"))

class SinPosEmb(nn.Module):
    def __init__(self,d): super().__init__(); self.d=d
    def forward(self,t):
        h=self.d//2
        f=torch.exp(-math.log(10000)*torch.arange(h,device=t.device)/(h-1))
        a=t[:,None].float()*f[None]
        return torch.cat([a.sin(),a.cos()],dim=-1)

class TEmb(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.sin = SinPosEmb(d)
        self.mlp = nn.Sequential(nn.Linear(d, d*4), nn.SiLU(), nn.Linear(d*4, d))
    def forward(self, t): return self.mlp(self.sin(t))

class SelfAttention(nn.Module):
    def __init__(self, ch, num_heads=4):
        super().__init__()
        while ch % num_heads != 0 and num_heads > 1: num_heads //= 2
        self.norm = nn.GroupNorm(32 if ch >= 32 else ch, ch)
        self.attn = nn.MultiheadAttention(ch, num_heads, batch_first=True)
        nn.init.zeros_(self.attn.out_proj.weight); nn.init.zeros_(self.attn.out_proj.bias)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).view(B, C, H * W).permute(0, 2, 1)
        h, _ = self.attn(h, h, h)
        h = h.permute(0, 2, 1).view(B, C, H, W)
        return x + h

class UNet(nn.Module):
    def __init__(self, in_channels=8, base_ch=160, ch_mult=(1,2,2,2), ze_dim=256):
        super().__init__()
        self.t_emb   = TEmb(ze_dim)
        chs          = [base_ch * m for m in ch_mult]
        self.in_proj = nn.Conv2d(in_channels, chs[0], 3, padding=1)
        self.enc_blocks  = nn.ModuleList(); self.downsamples = nn.ModuleList()
        prev = chs[0]
        for ch in chs[:-1]:
            self.enc_blocks.append(ResBlock(prev, ch, ze_dim))
            self.downsamples.append(Downsample(ch))
            prev = ch
        self.mid1 = ResBlock(prev, chs[-1], ze_dim)
        self.mid_attn = SelfAttention(chs[-1], num_heads=4)
        self.mid2 = ResBlock(chs[-1], chs[-1], ze_dim)
        prev = chs[-1]
        self.upsamples  = nn.ModuleList(); self.dec_blocks = nn.ModuleList()
        for ch in reversed(chs[:-1]):
            self.upsamples.append(Upsample(prev))
            self.dec_blocks.append(ResBlock(prev + ch, ch, ze_dim))
            prev = ch
        self.out_norm = nn.GroupNorm(32, prev)
        self.out_proj = nn.Conv2d(prev, 4, 3, padding=1)
        nn.init.zeros_(self.out_proj.weight); nn.init.zeros_(self.out_proj.bias)

    def forward(self, x, t, ze):
        cond = ze + self.t_emb(t); h = self.in_proj(x)
        skips = []
        for blk, down in zip(self.enc_blocks, self.downsamples):
            h = blk(h, cond); skips.append(h); h = down(h)
        h = self.mid1(h, cond); h = self.mid_attn(h); h = self.mid2(h, cond)
        for up, blk, sk in zip(self.upsamples, self.dec_blocks, reversed(skips)):
            h = up(h); h = torch.cat([h, sk], dim=1); h = blk(h, cond)
        return self.out_proj(F.silu(self.out_norm(h)))

class LP3(nn.Module):
    def __init__(self, in_dim=36, hidden_dim=512, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2), nn.GELU(),
            nn.Linear(hidden_dim // 2, out_dim)
        )
    def forward(self,x): return self.net(x)

class DDPM:
    def __init__(self,T=1000,device="cuda"):
        self.T=T
        t=torch.linspace(0,T,T+1,device=device)
        f=torch.cos(((t/T)+0.008)/1.008*math.pi/2)**2; f=f/f[0]
        ab=(f[1:]/f[:-1]).cumprod(0).clamp(1e-5,1-1e-5)
        self.alpha_bar=ab; self.sqrt_ab=ab.sqrt(); self.sqrt_one_m_ab=(1-ab).sqrt()
    def q_sample(self,x0,t,noise=None):
        if noise is None: noise=torch.randn_like(x0)
        return self.sqrt_ab[t][:,None,None,None]*x0+self.sqrt_one_m_ab[t][:,None,None,None]*noise,noise

class SidEmbed(nn.Module):
    def __init__(self, num_subjects=3, embed_dim=256):
        super().__init__()
        self.embed = nn.Embedding(num_subjects + 1, embed_dim)
    def forward(self, sid): return self.embed(sid)

# ── Load models ───────────────────────────────────────────────────────────────
print("\n  Loading Models (v15 160-Channel Architecture)...")
from diffusers import AutoencoderKL
vae=AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(DEVICE)
vae.eval()
for p in vae.parameters(): p.requires_grad_(False)

lp3=LP3().to(DEVICE)
sid_embed = SidEmbed().to(DEVICE)
unet=UNet(in_channels=8,base_ch=160).to(DEVICE)

def _strip_compiled(sd):
    return {k.replace("_orig_mod.", ""): v for k, v in sd.items()}

unet.load_state_dict(_strip_compiled(ckpt["unet"]), strict=True)
lp3.load_state_dict(_strip_compiled(ckpt["lp3"]), strict=True)
if "sid_embed" in ckpt:
    sid_embed.load_state_dict(_strip_compiled(ckpt["sid_embed"]), strict=True)

unet.eval(); lp3.eval(); sid_embed.eval()
ddpm=DDPM(T=1000,device=DEVICE)

print("  Loading LPIPS...")
lpips_fn=_LPIPS(net="vgg").to(DEVICE); lpips_fn.eval()
for p in lpips_fn.parameters(): p.requires_grad_(False)
print("  All models ready ✓")

@torch.no_grad()
def vae_encode(x): return vae.encode(x).latent_dist.sample()*0.18215
@torch.no_grad()
def vae_decode(z): return vae.decode(z/0.18215).sample.clamp(-1,1)

@torch.no_grad()
def ddim_sample(z1, ze_inp, sids, n_steps):
    ze_hdri = lp3(ze_inp)
    ze_sid  = sid_embed(sids.to(DEVICE))
    ze      = ze_hdri + ze_sid
    ts=torch.linspace(ddpm.T-1,0,n_steps,dtype=torch.long).tolist()
    x=torch.randn_like(z1)
    for i,t in enumerate(ts):
        t=int(t); tb=torch.full((z1.shape[0],),t,device=DEVICE,dtype=torch.long)
        eps=unet(torch.cat([x,z1],dim=1),tb,ze)
        ab=ddpm.alpha_bar[t]; x0=((x-(1-ab).sqrt()*eps)/ab.sqrt()).clamp(-1,1)
        if i<len(ts)-1:
            abp=ddpm.alpha_bar[int(ts[i+1])]; x=abp.sqrt()*x0+(1-abp).sqrt()*eps
        else: x=x0
    return x

# =============================================================================
# REMBG MASKS
# =============================================================================
def build_subject_masks(size=256):
    from rembg import remove as _rembg
    masks={}; sids=sorted(df_meta["subject_id"].unique())
    print(f"\n  Computing rembg masks for {len(sids)} subjects...")
    for sid in sids:
        sid=int(sid); rows=df_meta[df_meta["subject_id"]==sid]
        if not len(rows): continue
        try:
            pil=Image.open(rows.iloc[0]["image_path"]).convert("RGB").resize((size,size))
            alpha=(np.array(_rembg(pil))[:,:,3]/255.0>=0.5).astype(np.float32)
            masks[sid]=torch.from_numpy(alpha).unsqueeze(0).unsqueeze(0).to(DEVICE)
            print(f"    Subject {sid}: OK ({alpha.mean()*100:.1f}% fg)")
        except Exception as e:
            print(f"    Subject {sid}: rembg failed ({e}) -- ellipse fallback")
            H=W=size; cy,cx=H/2,W/2; ry,rx=H*.45,W*.40
            ys=torch.arange(H,device=DEVICE).float(); xs=torch.arange(W,device=DEVICE).float()
            yy,xx=torch.meshgrid(ys,xs,indexing="ij")
            masks[sid]=(((yy-cy)/ry)**2+((xx-cx)/rx)**2<=1).float().unsqueeze(0).unsqueeze(0)
    print("  Masks ready"); return masks

SUBJECT_MASKS=build_subject_masks(EVAL_CFG["image_size"])

def apply_mask(img,sid):
    m=SUBJECT_MASKS.get(int(sid))
    return img*m if m is not None else img

# =============================================================================
# HDRI BACKGROUND COMPOSITOR
# =============================================================================
_FOV_DEG      = 2*math.degrees(math.atan(36.0/(2*50.0)))  
_CAM_LOOK_AZ  = 45.0     
_CAM_LOOK_EL  = 2.9387   

def _aces_filmic(x):
    x = x.clip(0)
    a,b,c,d,e = 2.51, 0.03, 2.43, 0.59, 0.14
    t = (x*(a*x+b)) / (x*(c*x+d)+e)
    return np.power(t.clip(0,1), 1.0/2.2)

def hdri_to_background(hdr_npy, hdri_rotation_deg, camera_yaw_deg=-45.0, exposure_ev=0.0, out_size=256):
    hdr = ((hdr_npy.astype(np.float32)+1.0)/2.0).clip(0)
    _,H,W = hdr.shape
    hdr = hdr.transpose(1,2,0)  
    sample_az_deg = _CAM_LOOK_AZ - hdri_rotation_deg
    az_rad = np.deg2rad(sample_az_deg)
    el_rad = np.deg2rad(_CAM_LOOK_EL)
    focal = (out_size/2.0) / np.tan(np.deg2rad(_FOV_DEG/2.0))
    xs = np.arange(out_size)-out_size/2.0+0.5
    ys = np.arange(out_size)-out_size/2.0+0.5
    XX,YY = np.meshgrid(xs,ys)
    Xc=XX/focal; Yc=-YY/focal; Zc=np.ones_like(XX)
    n=np.sqrt(Xc**2+Yc**2+Zc**2); Xc/=n; Yc/=n; Zc/=n
    cp=np.cos(el_rad); sp=np.sin(el_rad)
    Xp=Xc; Yp=cp*Yc-sp*Zc; Zp=sp*Yc+cp*Zc
    ca=np.cos(az_rad); sa=np.sin(az_rad)
    Xw=ca*Xp+sa*Zp; Yw=Yp; Zw=-sa*Xp+ca*Zp
    az_eq  = np.arctan2(Xw,Zw)
    el_eq  = np.arcsin(np.clip(Yw,-1,1))
    u = (az_eq/(2*np.pi)+0.5)%1.0
    v = 0.5-el_eq/np.pi
    px=(u*W-0.5).clip(0,W-1); py=(v*H-0.5).clip(0,H-1)
    x0=np.floor(px).astype(int).clip(0,W-2); y0=np.floor(py).astype(int).clip(0,H-2)
    x1=x0+1; y1=y0+1
    wx=(px-x0)[...,None]; wy=(py-y0)[...,None]
    bg=(hdr[y0,x0]*(1-wx)*(1-wy) + hdr[y0,x1]*wx*(1-wy) +
        hdr[y1,x0]*(1-wx)*wy     + hdr[y1,x1]*wx*wy)
    bg = bg*(2.0**exposure_ev)
    bg = _aces_filmic(bg)
    return bg.astype(np.float32) 

def composite_face(face_tensor, mask_tensor, meta_row, size=256):
    """Composites a rendered face onto its exact matching HDRI background."""
    hdr_path = HDRI_CACHE / f"{_nk(str(meta_row['hdri_name_key']))}_hdr.npy"
    if not hdr_path.exists():
        return t2pil(face_tensor)
    
    hdr_npy = np.load(hdr_path)
    bg = hdri_to_background(
        hdr_npy,
        hdri_rotation_deg=float(meta_row.get("hdri_rotation_deg",0)),
        camera_yaw_deg=float(meta_row.get("camera_yaw_deg",-45)),
        exposure_ev=float(meta_row.get("exposure_used",0)),
        out_size=size)
    
    face_np = ((face_tensor.squeeze(0).permute(1,2,0).cpu().float().numpy()+1)/2).clip(0,1)
    mask_np = mask_tensor[0,0].cpu().float().numpy()
    comp = face_np * mask_np[:,:,None] + bg * (1-mask_np[:,:,None])
    return Image.fromarray((comp*255).clip(0,255).astype(np.uint8))

# =============================================================================
# DATASET
# =============================================================================

META_FIELDS = ["hdri_avg_lum","hdri_peak_lum","hdri_dyn_range","hdri_strength",
               "hdri_saturation","hdri_rotation_deg","exposure_used","hdri_tier"]

class RelightDataset(Dataset):
    def __init__(self,size=256,split="val",vf=0.02,seed=42):
        self.sz=size; self.df=df_meta.copy()
        if "hdri_name_key" not in self.df.columns:
            self.df["hdri_name_key"] = self.df["hdri_name"].astype(str).apply(_nk)
        self.mm,self.ms={},{}
        for f in META_FIELDS:
            if f in self.df.columns:
                v=self.df[f].astype(float)
                self.mm[f]=float(v.mean()); self.ms[f]=float(v.std())+1e-8
        self.sg={}
        for sid,g in self.df.groupby("subject_id"):
            self.sg[sid]=g.reset_index(drop=True)
        all_idx=[(s,i) for s,g in self.sg.items() for i in range(len(g))]
        r=random.Random(seed); r.shuffle(all_idx)
        nv=max(1,int(len(all_idx)*vf))
        raw=all_idx[:nv] if split=="val" else all_idx[nv:]
        self.idx=[(s,i) for s,i in raw
                  if _nk(str(self.sg[s].iloc[i]["hdri_name_key"])) in sh_index]
        self.tf=T.Compose([T.Resize((size,size)),T.ToTensor(),T.Normalize([.5]*3,[.5]*3)])

    def _sh(self,k):
        i=sh_index.get(_nk(str(k)))
        return sh_coeffs_all[i] if i is not None else np.zeros(27,dtype=np.float32)

    def _mv(self,row):
        vals=[]
        for f in META_FIELDS:
            if f=="hdri_rotation_deg":
                deg=float(row.get(f,0.0))
                vals.append(math.sin(math.radians(deg)))
                vals.append(math.cos(math.radians(deg)))
            else:
                v=(float(row.get(f,0))-self.mm.get(f,0))/self.ms.get(f,1)
                vals.append(max(-5.0,min(5.0,v)))
        return np.array(vals,dtype=np.float32)

    def _img(self,p): return self.tf(Image.open(p).convert("RGB"))

    def __len__(self): return len(self.idx)

    def __getitem__(self,idx):
        sid,i=self.idx[idx]; g=self.sg[sid]; r1=g.iloc[i]
        c=g[g["hdri_rank"]!=r1["hdri_rank"]]; r2=(c if len(c) else g).sample(1).iloc[0]
        sh2=self._sh(r2["hdri_name_key"])
        ze=torch.from_numpy(np.concatenate([sh2,self._mv(r2)]))
        # RETURNING RAW DICTS FOR METADATA SO WE CAN COMPOSITE LATER
        return (self._img(r1["image_path"]),self._img(r2["image_path"]),
                ze,torch.tensor(int(sid),dtype=torch.long),
                r1.to_dict(), r2.to_dict())

print("\n  Building val dataset...")
val_ds=RelightDataset(size=EVAL_CFG["image_size"],split="val",
                      vf=EVAL_CFG["val_fraction"],seed=EVAL_CFG["val_seed"])
# We use a custom collate_fn because dicts are returned
def custom_collate(batch):
    i1 = torch.stack([item[0] for item in batch])
    i2 = torch.stack([item[1] for item in batch])
    ze = torch.stack([item[2] for item in batch])
    sid = torch.stack([item[3] for item in batch])
    r1 = [item[4] for item in batch]
    r2 = [item[5] for item in batch]
    return i1, i2, ze, sid, r1, r2

n_eval=min(EVAL_CFG["n_eval"],len(val_ds))
val_loader=DataLoader(val_ds,batch_size=EVAL_CFG["batch_size"],
                      shuffle=False,num_workers=2,pin_memory=True,
                      collate_fn=custom_collate)
print(f"  Val: {len(val_ds)} samples  (evaluating {n_eval})")

# =============================================================================
# METRIC HELPERS
# =============================================================================
def t2np(t): return ((t.squeeze(0).clamp(-1,1)+1)/2).permute(1,2,0).cpu().float().numpy()
def t2pil(t): return Image.fromarray((t2np(t)*255).clip(0,255).astype(np.uint8))
def psnr(p,g): m=np.mean((p-g)**2); return 100. if m<1e-10 else 10*math.log10(1/m)
def ssim(p,g): return float(_ssim_fn(p,g,data_range=1.,channel_axis=-1,win_size=7))
def pmse(p,g): return float(np.mean((p-g)**2))

# =============================================================================
# PART A -- QUANTITATIVE METRICS
# =============================================================================

print("\n"+"="*65)
print(f"  PART A -- Metrics  ({n_eval} samples, {EVAL_CFG['ddim_steps']} DDIM steps)")
print("="*65)

all_m=[]; done=0; t0=time.time(); samps = []

for img1b,img2b,zeb,sidb,r1b,r2b in val_loader:
    if done>=n_eval: break
    B=min(img1b.shape[0],n_eval-done)
    i1=img1b[:B].to(DEVICE); i2=img2b[:B].to(DEVICE)
    ze=zeb[:B].to(DEVICE);   si=sidb[:B]
    with torch.no_grad():
        i1m=torch.stack([apply_mask(i1[k:k+1],si[k].item())[0] for k in range(B)])
        i2m=torch.stack([apply_mask(i2[k:k+1],si[k].item())[0] for k in range(B)])
        z1=vae_encode(i1m); z2=vae_encode(i2m)
        tr=torch.randint(0,1000,(B,),device=DEVICE)
        xt,eps=ddpm.q_sample(z2,tr)
        zev=lp3(ze); ep=unet(torch.cat([xt,z1],dim=1),tr,zev)
        lmse=F.mse_loss(ep,eps).item()
        zp=ddim_sample(z1,ze,si,EVAL_CFG["ddim_steps"])
        pred=vae_decode(zp)
        lp=lpips_fn(pred,i2m).squeeze()
        if lp.ndim==0: lp=lp.unsqueeze(0)
    for k in range(B):
        pn=t2np(pred[k]); gn=t2np(i2m[k])
        m={"idx":done+k,"lmse":float(lmse),"pmse":pmse(pn,gn),
           "psnr":psnr(pn,gn),"ssim":ssim(pn,gn),"lpips":float(lp[k].item())}
        all_m.append(m)
        
        # ── THE COMPOSITING FIX ──
        # Instead of saving the raw face, we composite it with the exact background
        mask_t = SUBJECT_MASKS.get(int(si[k].item()))
        pred_comp_img = composite_face(pred[k:k+1], mask_t, r2b[k], size=EVAL_CFG["image_size"])
        gt_comp_img   = composite_face(i2m[k:k+1],  mask_t, r2b[k], size=EVAL_CFG["image_size"])
        
        samps.append((t2pil(i1m[k]), gt_comp_img, pred_comp_img, m))
        print(f"  [{done+k+1:3d}/{n_eval}]  PSNR={m['psnr']:5.2f}  SSIM={m['ssim']:.3f}"
              f"  LPIPS={m['lpips']:.3f}  MSE={m['pmse']:.4f}  ({time.time()-t0:.0f}s)")
    done+=B

def avg(k): return float(np.mean([m[k] for m in all_m]))
def std(k): return float(np.std([m[k] for m in all_m]))

print("\n"+"="*65)
print(f"  SUMMARY  step={ckpt_step}  n={n_eval}")
print("="*65)
for name,k in [("Pixel MSE", "pmse"),("PSNR dB", "psnr"),("SSIM", "ssim"),("LPIPS-VGG", "lpips")]:
    print(f"  {name:<12} mean={avg(k):>8.4f}  std={std(k):>7.4f}")

with open(EVAL_DIR/"eval_metrics.json","w") as f:
    json.dump({"ckpt":best_path.name,"step":str(ckpt_step),"n":n_eval,
               "summary":{k:{"mean":avg(k),"std":std(k)} for k in ["pmse","psnr","ssim","lpips"]},
               "samples":all_m},f,indent=2)

# =============================================================================
# PART B -- VISUAL COMPARISON GRIDS
# =============================================================================
print("\n"+"─"*65)
print("  PART B -- Standard Eval Grids")
print("─"*65)

NC = EVAL_CFG["grid_cols"]; SPG = EVAL_CFG["samples_per_grid"]; N_GRIDS = EVAL_CFG["n_grids"]
SZ = EVAL_CFG["image_size"]; PAD = 4; HDR = 22; SCH = 28
BW = SZ*3 + PAD*2 + PAD; CW = BW*NC + PAD; NR = math.ceil(SPG / NC)
CH = HDR + PAD + NR*(SZ + SCH + PAD)

try:
    fnt  = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", 9)
    fnt2 = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", 11)
except: fnt = fnt2 = ImageFont.load_default()

GDIR = EVAL_DIR / "grids"; GDIR.mkdir(exist_ok=True)
grid_paths = []
for g in range(N_GRIDS):
    batch = samps[g*SPG : (g+1)*SPG]
    if not batch: break
    canvas = Image.new("RGB", (CW, CH), (12, 12, 12))
    draw   = ImageDraw.Draw(canvas)
    for ci in range(min(NC, len(batch))):
        xb = PAD + ci*BW
        for li, (lb, col) in enumerate(zip(["Input", "Ground Truth", "Predicted"],
                                           [(100,100,100), (70,185,70), (70,145,215)])):
            draw.text((xb + li*(SZ+PAD) + 2, 4), lb, fill=col, font=fnt)
    for si2, (inp, gt, pred, m) in enumerate(batch):
        ri = si2 // NC; ci = si2 % NC
        xo = PAD + ci*BW; yo = HDR + PAD + ri*(SZ + SCH + PAD)
        canvas.paste(inp,  (xo,             yo))
        canvas.paste(gt,   (xo+SZ+PAD,      yo))
        canvas.paste(pred, (xo+SZ*2+PAD*2,  yo))
        sc = (65,200,65) if m["psnr"]>25 else (210,190,50) if m["psnr"]>20 else (210,65,65)
        draw.text((xo+2, yo+SZ+4), f"s{m['idx']:02d} PSNR{m['psnr']:.1f} SSIM{m['ssim']:.2f} LPIPS{m['lpips']:.3f}", fill=sc, font=fnt)
    gp = GDIR / f"eval_grid_{g+1:02d}.png"
    canvas.save(gp); grid_paths.append(gp)
    print(f"  Grid {g+1:02d} saved → {gp.name}")

# =============================================================================
# PART C -- ROTATION ANALYSIS GRIDS (Perfect 360° Sweep)
# =============================================================================
print("\n"+"─"*65)
print("  PART C -- Rotation Analysis Grids")
print("─"*65)

# 1. Find groups where the same Subject + HDRI has the full 45-angle sweep
rot_groups = []
for (sid, hkey), grp in df_meta.groupby(["subject_id", "hdri_name_key"]):
    # Look for groups that have the full ~45 angles
    if len(grp["hdri_rotation_deg"].unique()) > 10: 
        rot_groups.append((sid, hkey, grp))

random.seed(EVAL_CFG["val_seed"])
random.shuffle(rot_groups)
n_rot_grids = min(EVAL_CFG["n_rotation_grids"], len(rot_groups))

ROT_DIR = EVAL_DIR / "rotation_grids"; ROT_DIR.mkdir(exist_ok=True)
rot_paths = []

# We will sample 5 angles across the 45-angle sweep to show a full rotation
NUM_ANGLES_TO_SHOW = 5

for idx in range(n_rot_grids):
    sid, hkey, grp = rot_groups[idx]
    
    # 2. Get all 45 unique angles, sorted mathematically
    all_angles = sorted(grp["hdri_rotation_deg"].unique())
    
    # 3. Evenly sample indices across the 45 steps (e.g., indices 0, 11, 22, 33, 44)
    sample_indices = np.linspace(0, len(all_angles)-1, NUM_ANGLES_TO_SHOW, dtype=int)
    sweep_angles = [all_angles[i] for i in sample_indices]
    
    cols = len(sweep_angles)
    
    # Use one common Input image for this subject
    input_row = df_meta[df_meta["subject_id"] == sid].iloc[0]
    i1_img = val_ds._img(input_row["image_path"]).unsqueeze(0).to(DEVICE)
    i1_mask = apply_mask(i1_img, sid)
    z1 = vae_encode(i1_mask)
    mask_t = SUBJECT_MASKS.get(int(sid))
    
    # Create Canvas: Cols = Angles, Rows = 3 (Input, GT, Pred)
    r_cw = PAD + cols*(SZ+PAD)
    r_ch = HDR + PAD + 3*(SZ+PAD)
    rcanvas = Image.new("RGB", (r_cw, r_ch), (12,12,12))
    rdraw = ImageDraw.Draw(rcanvas)
    
    rdraw.text((PAD, 4), f"Subject {sid}  |  HDRI: {hkey}  |  360° Orbit (8° increments)", fill=(200,200,200), font=fnt2)
    
    # Process each selected angle in the sweep
    for c, angle in enumerate(sweep_angles):
        target_row = grp[grp["hdri_rotation_deg"] == angle].iloc[0]
        
        # Row 1: Input Image (Constant)
        inp_pil = t2pil(i1_mask)
        
        # Row 2: GT Image (Composited Background)
        i2_img = val_ds._img(target_row["image_path"]).unsqueeze(0).to(DEVICE)
        i2_mask = apply_mask(i2_img, sid)
        gt_comp = composite_face(i2_mask, mask_t, target_row.to_dict(), size=SZ)
        
        # Row 3: Predict & Composite Background
        sh2 = val_ds._sh(target_row["hdri_name_key"])
        ze_inp = torch.from_numpy(np.concatenate([sh2, val_ds._mv(target_row.to_dict())])).unsqueeze(0).to(DEVICE)
        s_tensor = torch.tensor([int(sid)], dtype=torch.long).to(DEVICE)
        
        with torch.no_grad():
            zp = ddim_sample(z1, ze_inp, s_tensor, EVAL_CFG["ddim_steps"])
            pred = vae_decode(zp)
            
        pred_comp = composite_face(pred, mask_t, target_row.to_dict(), size=SZ)
        
        # Draw onto the grid
        xo = PAD + c*(SZ+PAD)
        rdraw.text((xo+2, HDR+PAD+2), f"Angle: {angle:.1f}°", fill=(255,200,80), font=fnt)
        rcanvas.paste(inp_pil,   (xo, HDR+PAD+20))              # Row 1
        rcanvas.paste(gt_comp,   (xo, HDR+PAD*2+SZ+20))         # Row 2
        rcanvas.paste(pred_comp, (xo, HDR+PAD*3+SZ*2+20))       # Row 3
        
    rp = ROT_DIR / f"rot_grid_s{sid}_{hkey}.png"
    rcanvas.save(rp); rot_paths.append(rp)
    print(f"  Rotation Grid {idx+1}/{n_rot_grids} saved → {rp.name}")

# ── ZIP everything ────────────────────────────────────────────────────────────
ZP = WORK / "relight_v15_eval.zip"
with zipfile.ZipFile(ZP, "w", zipfile.ZIP_DEFLATED) as zf:
    # Safely zip standard grids if they exist
    for p in grid_paths: 
        if p.exists(): zf.write(p, f"standard_grids/{p.name}")
    # Safely zip rotation grids if they exist
    for p in rot_paths:  
        if p.exists(): zf.write(p, f"rotation_grids/{p.name}")
    # Write metrics JSON
    json_path = EVAL_DIR / "eval_metrics.json"
    if json_path.exists():
        zf.write(json_path, "eval_metrics.json")

print(f"\n  ZIP saved: {ZP}  ({ZP.stat().st_size/1e6:.1f} MB)  <- DOWNLOAD THIS")

print("="*65)
print(f"  DONE  --  step={ckpt_step}  |  Ready for download.")
print("="*65)

In [ ]:
import torch
ck = torch.load("/kaggle/working/checkpoints/last.pt", map_location="cpu")
print(f"step={ck['step']}  best_loss={ck['best_loss']}")
# Should print: step=5000  best_loss=...

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image

OUT_DIR = Path("/kaggle/working/masks_output")
OUT_DIR.mkdir(exist_ok=True)

IMAGE_SIZE     = 256
EROSION_KERNEL = 5

ALEXANDER_ROOT = Path("/kaggle/input/datasets/devsachaniitk/alexander/alexander")

VERSION_TO_SUBJECT = {
    "synthlight_renders_v6": 1,
    "synthlight_renders_v7": 2,
    "synthlight_renders_v8": 3,
}

def erode_mask_numpy(mask, kernel=5):
    """Pure numpy erosion — no scipy needed."""
    pad = kernel // 2
    eroded = mask.copy()
    padded = np.pad(mask, pad, mode='constant', constant_values=0)
    for i in range(mask.shape[0]):
        for j in range(mask.shape[1]):
            patch = padded[i:i+kernel, j:j+kernel]
            eroded[i, j] = 1.0 if patch.min() > 0.5 else 0.0
    return eroded.astype(np.float32)

for version, sid in VERSION_TO_SUBJECT.items():
    mesh_dir = ALEXANDER_ROOT / version / "model_mesh"
    if not mesh_dir.exists():
        print(f"[SKIP] Subject {sid} — {mesh_dir} not found")
        continue

    img_files = sorted(mesh_dir.glob("*.png")) + sorted(mesh_dir.glob("*.jpg"))
    if not img_files:
        print(f"[SKIP] Subject {sid} — no images found")
        continue

    img_path = img_files[len(img_files) // 2]
    print(f"Subject {sid} — using: {img_path.name}")

    pil = Image.open(img_path)
    pil_resized = pil.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)

    if pil.mode == "RGBA":
        alpha = np.array(pil_resized)[:, :, 3].astype(np.float32) / 255.0
        mask_hard = (alpha > 0.6).astype(np.float32)
        print(f"  Used RGBA alpha")
    else:
        rgb = np.array(pil_resized).astype(np.float32) / 255.0
        r, g, b = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
        max_ch = np.maximum(np.maximum(r, g), b)
        min_ch = np.minimum(np.minimum(r, g), b)
        saturation = (max_ch - min_ch) / (max_ch + 1e-8)
        mask_hard = (saturation > 0.05).astype(np.float32)
        print(f"  Used saturation threshold")

    print(f"  Eroding mask (pure numpy, kernel={EROSION_KERNEL})...")
    mask_eroded = erode_mask_numpy(mask_hard, EROSION_KERNEL)

    out_path = OUT_DIR / f"mask_subject{sid}.npy"
    np.save(str(out_path), mask_eroded)
    print(f"  Saved {out_path.name}  fg={mask_eroded.mean():.2%}\n")

print("Done. Files:")
for f in sorted(OUT_DIR.glob("*.npy")):
    print(f"  {f.name}  shape={np.load(f).shape}")

In [ ]:
subprocess.run(["pip", "install", "rembg[cpu]", "--quiet"], check=True)

In [ ]:
"""
relight_viz_cell_v2.py  [VISUALIZATION CELL — v2 FIXED & CORRECTED]
==================================================================
FIXES APPLIED:
  [FIX-V1]  Angle group collapse — create hdri_name_key, handle missing columns.
  [FIX-V2]  Illum diff map with foreground mask (rembg).
  [FIX-V3]  SH→Face map with stronger normals & tone mapping.
  [FIX-V4]  Masked loss diagnostics.
  [FIX-V5]  CSV structure inspector.
  [FIX-V6]  Added missing hdri_name_key and hdri_rank columns.
"""
import pandas as pd
import os
import sys
import json
import math
import random
import shutil
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

# ─────────────────────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────────────────────

KAGGLE_DATA_ROOT = Path("/kaggle/input")
KAGGLE_WORK_DIR  = Path("/kaggle/working")

def find_alexander_root():
    candidates = [
        KAGGLE_DATA_ROOT / "alexander" / "alexander",
        KAGGLE_DATA_ROOT / "alexander",
        KAGGLE_DATA_ROOT / "relight-dataset" / "alexander",
        KAGGLE_DATA_ROOT / "synthlight" / "alexander",
        KAGGLE_DATA_ROOT / "datasets/devsachaniitk/alexander/alexander",
        Path("D:/alexander"),
    ]
    for c in candidates:
        if c.exists():
            print(f"  [PATH] Data root: {c}")
            return c
    for p in KAGGLE_DATA_ROOT.rglob("synthlight_renders_v6"):
        return p.parent
    raise FileNotFoundError("Could not find alexander/ folder.")

ALEXANDER_ROOT = find_alexander_root()

HDRI_CACHE_DIR = Path("/kaggle/input/datasets/devsachaniitk/hdri-cache/hdri_cache")
if not HDRI_CACHE_DIR.exists():
    found = list(Path("/kaggle/input").rglob("hdri_cache"))
    HDRI_CACHE_DIR = found[0] if found else HDRI_CACHE_DIR

SH_CACHE_DIR = KAGGLE_WORK_DIR / "sh_cache"
CSV_PATH     = KAGGLE_WORK_DIR / "master_metadata.csv"
CKPT_DIR     = KAGGLE_WORK_DIR / "checkpoints"
VIZ_DIR      = KAGGLE_WORK_DIR / "viz_output_v2"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  [DEVICE] {DEVICE}")


# ─────────────────────────────────────────────────────────────
# SECTION 0 — CSV STRUCTURE INSPECTOR  [FIX-V5]
# ─────────────────────────────────────────────────────────────

def inspect_csv_structure(csv_path: Path) -> dict:
    import pandas as pd

    print(f"\n{'='*60}")
    print("  CSV STRUCTURE INSPECTOR  [FIX-V5]")
    print(f"{'='*60}")

    df = pd.read_csv(csv_path)
    print(f"\n  Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\n  ALL COLUMNS:")
    for col in df.columns:
        dtype = df[col].dtype
        n_unique = df[col].nunique()
        sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else "N/A"
        print(f"    {col:<30} dtype={str(dtype):<10} unique={n_unique:<8} sample={str(sample)[:40]}")

    angle_candidates = [c for c in df.columns if any(
        kw in c.lower() for kw in ["yaw", "angle", "azimuth", "camera", "pose", "rot", "deg"]
    )]
    print(f"\n  ANGLE-RELATED COLUMNS: {angle_candidates}")

    for col in angle_candidates:
        vals = df[col].dropna()
        print(f"\n  [{col}]")
        print(f"    unique values : {vals.nunique()}")
        # Check if column is numeric before printing min/max
        if pd.api.types.is_numeric_dtype(vals):
            print(f"    min/max       : {vals.min():.2f} / {vals.max():.2f}")
        else:
            print(f"    type          : non-numeric (string/categorical)")
        print(f"    value counts  :")
        vc = vals.value_counts().head(20)
        for v, cnt in vc.items():
            # Format numeric values nicely, else string
            if pd.api.types.is_numeric_dtype(vals):
                print(f"      {v:>10.2f}° → {cnt:,} images")
            else:
                print(f"      {str(v):<20} → {cnt:,} images")

    # Subject breakdown
    print(f"\n  SUBJECT BREAKDOWN:")
    for sid, grp in df.groupby("subject_id"):
        yaw_col = "camera_yaw_deg" if "camera_yaw_deg" in df.columns else angle_candidates[0] if angle_candidates else None
        if yaw_col:
            n_yaws = grp[yaw_col].nunique()
            n_hdris = grp["hdri_name"].nunique() if "hdri_name" in df.columns else "?"
            print(f"    Subject {sid}: {len(grp):,} rows | {n_yaws} unique yaws | {n_hdris} unique HDRIs")
        else:
            print(f"    Subject {sid}: {len(grp):,} rows")

    yaw_col = "camera_yaw_deg" if "camera_yaw_deg" in df.columns else None
    findings = {"yaw_col": yaw_col, "n_unique_yaws": 0, "yaw_values": [], "is_collapsed": True}
    if yaw_col:
        unique_yaws = sorted(df[yaw_col].unique())
        findings["n_unique_yaws"] = len(unique_yaws)
        findings["yaw_values"]    = unique_yaws
        findings["is_collapsed"]  = len(unique_yaws) <= 5
        print(f"\n  YAW COLUMN ANALYSIS: '{yaw_col}'")
        print(f"    Unique values: {len(unique_yaws)}")
        print(f"    Values: {unique_yaws[:20]}{'...' if len(unique_yaws) > 20 else ''}")

        if len(unique_yaws) <= 3:
            print(f"\n  !! PROBLEM DETECTED: Only {len(unique_yaws)} unique yaw values !!")
            print(f"     Expected ~45 unique values (e.g., 0,8,16,...352 degrees)")
            print(f"     This means angle grouping collapsed all images into {len(unique_yaws)} groups")
            print(f"\n  LIKELY CAUSES:")
            print(f"     A) camera_yaw_deg was set to 0 for all rows during CSV build")
            print(f"        → Fix: check json entries for 'camera_yaw_deg' field name")
            print(f"     B) The JSON metadata uses a different field name")
            print(f"        → Fix: inspect raw JSON, find correct field")
            print(f"     C) Yaw is encoded in the filename instead of metadata")
            print(f"        → Fix: parse yaw from filename")

            if "filename" in df.columns:
                sample_fns = df["filename"].dropna().head(10).tolist()
                print(f"\n  FILENAME SAMPLES (looking for angle info):")
                for fn in sample_fns:
                    print(f"    {fn}")
                print(f"\n  → If filenames contain angle (e.g. 'render_yaw045_hdri.png')")
                print(f"    we can extract it via regex.")

    # Inspect raw JSON
    print(f"\n  INSPECTING RAW JSON METADATA (first subject):")
    VERSION_TO_SUBJECT = {"synthlight_renders_v6": 1, "synthlight_renders_v7": 2,
                          "synthlight_renders_v8": 3, "synthlight_renders_v9_cache_128": 4}
    for version, sid in VERSION_TO_SUBJECT.items():
        mesh_dir = ALEXANDER_ROOT / version / "model_mesh"
        if not mesh_dir.exists(): continue
        jsons = list(mesh_dir.glob("*.json"))
        if not jsons: continue
        with open(jsons[0]) as f:
            entries = json.load(f)
        if isinstance(entries, dict):
            entries = next(v for v in entries.values() if isinstance(v, list))

        print(f"\n  Subject {sid} — {jsons[0].name}")
        print(f"  First 3 entries:")
        for e in entries[:3]:
            print(f"    Keys: {list(e.keys())}")
            for k, v in e.items():
                if any(kw in k.lower() for kw in ["yaw","angle","camera","pose","azimuth","rot"]):
                    print(f"      *** ANGLE FIELD: {k} = {v}")
            print()
        break

    return findings

# ─────────────────────────────────────────────────────────────
# SECTION 1 — CHECKPOINT VERIFIER
# ─────────────────────────────────────────────────────────────

def verify_checkpoint(ckpt_path: Path) -> dict:
    print(f"\n{'='*60}")
    print(f"  CHECKPOINT VERIFIER: {ckpt_path.name}")
    print(f"{'='*60}")

    if not ckpt_path.exists():
        print(f"  [FAIL] Not found: {ckpt_path}")
        return {"valid": False, "step": 0}

    try:
        ckpt = torch.load(ckpt_path, map_location="cpu")
    except Exception as e:
        print(f"  [FAIL] Cannot load: {e}")
        return {"valid": False, "step": 0}

    step      = ckpt.get("step", 0)
    best_loss = ckpt.get("best_loss", float("inf"))
    print(f"  Step       : {step:,}")
    print(f"  Best loss  : {best_loss:.6f}")
    print(f"  Keys found : {list(ckpt.keys())}")

    report = {"valid": True, "step": step, "best_loss": best_loss,
              "nan_keys": [], "weight_stats": {}}

    for model_key in ["unet", "lp3"]:
        state = ckpt.get(model_key, {})
        if not state: continue
        total_params = 0; nan_count = 0; inf_count = 0
        all_means = []; all_stds = []
        for k, v in state.items():
            if not torch.is_tensor(v): continue
            total_params += v.numel()
            if torch.isnan(v).any():  nan_count += v.numel()
            if torch.isinf(v).any():  inf_count += v.numel()
            if v.numel() > 1:
                all_means.append(float(v.float().mean()))
                all_stds.append(float(v.float().std()))
        status = "✓ CLEAN" if (nan_count == 0 and inf_count == 0) else "✗ CORRUPT"
        print(f"\n  [{model_key.upper()}]  {status}")
        print(f"    Total params : {total_params:,}")
        print(f"    NaN params   : {nan_count}")
        print(f"    Inf params   : {inf_count}")
        if all_means:
            print(f"    Weight mean  : {np.mean(all_means):.6f}")
            print(f"    Weight std   : {np.mean(all_stds):.6f}")
        if nan_count > 0 or inf_count > 0:
            report["valid"] = False

    verdict = "✓ VALID" if report["valid"] else "✗ CORRUPT"
    print(f"\n  VERDICT: {verdict} — step {step:,}")
    return report


# ─────────────────────────────────────────────────────────────
# SECTION 2 — FIXED DATASET  [FIX-V1 + FIX-V6]
# ─────────────────────────────────────────────────────────────

def _normalise_hdri_key(raw: str) -> str:
    key = raw.strip().lower()
    for ext in (".hdr", ".exr", ".png"):
        if key.endswith(ext):
            key = key[:-len(ext)]; break
    return key

def detect_yaw_column(df) -> str:
    import pandas as pd
    angle_candidates = [c for c in df.columns if any(
        kw in c.lower() for kw in ["yaw", "angle", "azimuth"]
    )]
    if not angle_candidates:
        return None
    best_col, best_unique = None, 0
    for col in angle_candidates:
        n = df[col].nunique()
        if n > best_unique:
            best_unique = n; best_col = col
    return best_col

def try_extract_yaw_from_filename(filename: str) -> float:
    import re
    patterns = [
        r'yaw[_\-]?(\d+)',
        r'(\d+)deg',
        r'angle[_\-]?(\d+)',
        r'_(\d{3})[_\.]',
        r'^(\d{3,4})[_\.]',
    ]
    for pat in patterns:
        m = re.search(pat, str(filename).lower())
        if m:
            return float(m.group(1))
    return -1.0

class RelightDatasetV2(Dataset):
    def __init__(self, csv_path, sh_npy_path, sh_index_path,
                 image_size=256, split="train", val_fraction=0.02, seed=42,
                 force_yaw_col=None):
        import pandas as pd
        self.image_size = image_size

        df = pd.read_csv(csv_path)

        # [FIX-V6] Ensure required columns exist
        if "hdri_name" not in df.columns:
            raise KeyError("CSV missing required column 'hdri_name'")
        # Create normalised key for HDRI matching
        df["hdri_name_key"] = df["hdri_name"].astype(str).apply(_normalise_hdri_key)

        # If hdri_rank missing, create a dummy rank per (subject, hdri)
        if "hdri_rank" not in df.columns:
            print("  [DatasetV2] 'hdri_rank' column not found. Creating dummy ranks.")
            df["hdri_rank"] = df.groupby(["subject_id", "hdri_name"]).cumcount()

        self.sh_coeffs = np.load(sh_npy_path)
        with open(sh_index_path) as f:
            self.sh_index = {_normalise_hdri_key(k): v
                             for k, v in json.load(f).items()}

        # Metadata fields
        METADATA_FIELDS = [
            "hdri_avg_lum","hdri_peak_lum","hdri_dyn_range","hdri_strength",
            "hdri_saturation","hdri_rotation_deg","exposure_used","hdri_tier",
        ]
        self.meta_mean = {}; self.meta_std = {}
        for field in METADATA_FIELDS:
            if field in df.columns:
                vals = df[field].astype(float)
                self.meta_mean[field] = float(vals.mean())
                self.meta_std[field]  = float(vals.std()) + 1e-8
        self.METADATA_FIELDS = METADATA_FIELDS

        # Detect yaw column
        if force_yaw_col and force_yaw_col in df.columns:
            yaw_col = force_yaw_col
            print(f"  [DatasetV2] Using forced yaw column: '{yaw_col}'")
        else:
            yaw_col = detect_yaw_column(df)
            print(f"  [DatasetV2] Auto-detected yaw column: '{yaw_col}'")

        if yaw_col and df[yaw_col].nunique() >= 10:
            df["_yaw_key"] = df[yaw_col].round(1)
            print(f"  [DatasetV2] Yaw column '{yaw_col}': {df['_yaw_key'].nunique()} unique values ✓")
        else:
            print(f"  [DatasetV2] Yaw column collapsed/missing — extracting from filename")
            if "filename" in df.columns:
                df["_yaw_key"] = df["filename"].apply(try_extract_yaw_from_filename)
                n_extracted = (df["_yaw_key"] >= 0).sum()
                n_unknown   = (df["_yaw_key"] < 0).sum()
                print(f"  [DatasetV2] Extracted yaw from filename: {n_extracted} ok, {n_unknown} unknown")
                if n_extracted < len(df) * 0.5:
                    print(f"  [DatasetV2] !! Less than 50% extracted — check filename patterns !!")
                    df["_yaw_key"] = 0.0
            else:
                print(f"  [DatasetV2] No filename column — subject-only grouping")
                df["_yaw_key"] = 0.0

        # Build angle groups: (subject_id, _yaw_key)
        self.angle_groups = {}
        for (sid, yaw), grp in df.groupby(["subject_id", "_yaw_key"]):
            rows = grp.reset_index(drop=True)
            valid_rows = rows[
                rows["hdri_name_key"].apply(
                    lambda k: k in self.sh_index
                )
            ].reset_index(drop=True)
            if len(valid_rows) >= 1:
                self.angle_groups[(int(sid), float(yaw))] = valid_rows

        n_groups = len(self.angle_groups)
        n_rows   = sum(len(v) for v in self.angle_groups.values())
        single   = sum(1 for v in self.angle_groups.values() if len(v) == 1)
        multi    = n_groups - single

        print(f"  [DatasetV2] Angle groups: {n_groups}  (expected ~{45 * df['subject_id'].nunique()})")
        print(f"  [DatasetV2] Valid rows  : {n_rows:,}")
        print(f"  [DatasetV2] True pairs (≥2 HDRIs per angle): {multi} groups")
        print(f"  [DatasetV2] Fallback    (1 HDRI per angle) : {single} groups")

        if n_groups <= 10:
            print(f"\n  !! WARNING: Only {n_groups} angle groups found !!")
            print(f"     This means the yaw column is still collapsed.")
            print(f"     Run inspect_csv_structure() to diagnose — check the")
            print(f"     raw JSON metadata for the correct angle field name.")
            print(f"     Then pass force_yaw_col='your_col' to this dataset.\n")

        # Train/val split
        all_keys = list(self.angle_groups.keys())
        rng = random.Random(seed); rng.shuffle(all_keys)
        n_val = max(1, int(len(all_keys) * val_fraction))
        split_keys = all_keys[n_val:] if split == "train" else all_keys[:n_val]

        self.indices = [
            (key, i)
            for key in split_keys
            for i in range(len(self.angle_groups[key]))
        ]
        print(f"  [DatasetV2/{split}] {len(self.indices):,} samples across {len(split_keys)} angle groups")

        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])

    def _get_sh(self, key):
        norm_key = _normalise_hdri_key(str(key))
        idx = self.sh_index.get(norm_key)
        return np.zeros(27, dtype=np.float32) if idx is None else np.clip(self.sh_coeffs[idx], -10.0, 10.0)

    def _get_meta(self, row):
        vals = []
        for f in self.METADATA_FIELDS:
            if f not in row.index:
                vals.append(0.0)
                continue
            if f == "hdri_rotation_deg":
                raw_deg = float(row[f]) if not pd.isna(row[f]) else 0.0
                vals += [math.sin(math.radians(raw_deg)), math.cos(math.radians(raw_deg))]
            else:
                v = (float(row[f]) - self.meta_mean.get(f, 0)) / self.meta_std.get(f, 1)
                vals.append(float(np.clip(v, -5.0, 5.0)))
        return np.array(vals, dtype=np.float32)

    def _load_img(self, path):
        try:
            img = self.transform(Image.open(path).convert("RGB"))
            return img if not (torch.isnan(img).any() or torch.isinf(img).any()) \
                       else torch.zeros(3, self.image_size, self.image_size)
        except Exception:
            return torch.zeros(3, self.image_size, self.image_size)

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        (sid, yaw), row_i = self.indices[idx]
        grp  = self.angle_groups[(sid, yaw)]
        row1 = grp.iloc[row_i]

        if len(grp) > 1:
            # Choose another row with different HDRI
            other = grp[grp["hdri_rank"] != row1["hdri_rank"]]
            if len(other) == 0:
                other = grp.drop(row_i)
            if len(other) == 0:
                row2 = row1  # fallback to identity
            else:
                row2 = other.sample(1).iloc[0]
        else:
            row2 = row1

        img1 = self._load_img(row1["image_path"])
        img2 = self._load_img(row2["image_path"])
        sh1  = self._get_sh(row1["hdri_name_key"])
        sh2  = self._get_sh(row2["hdri_name_key"])
        meta2 = self._get_meta(row2)
        ze_input = torch.from_numpy(np.concatenate([sh2, meta2])).clamp(-10.0, 10.0)

        return {
            "img_src"    : img1,
            "img_tgt"    : img2,
            "ze_input"   : ze_input,
            "sh_src"     : torch.from_numpy(sh1),
            "sh_tgt"     : torch.from_numpy(sh2),
            "subject_id" : torch.tensor(sid, dtype=torch.long),
            "camera_yaw" : torch.tensor(yaw, dtype=torch.float32),
            "src_path"   : str(row1["image_path"]),
            "tgt_path"   : str(row2["image_path"]),
            "src_hdri"   : str(row1["hdri_name"]),
            "tgt_hdri"   : str(row2["hdri_name"]),
            "is_identity": torch.tensor(
                str(row1["hdri_name"]) == str(row2["hdri_name"]), dtype=torch.bool),
        }


# ─────────────────────────────────────────────────────────────
# SECTION 3 — FOREGROUND MASK (rembg)  [FIX-V2]
# ─────────────────────────────────────────────────────────────

_rembg_fn = None

def get_rembg():
    global _rembg_fn
    if _rembg_fn is None:
        try:
            from rembg import remove as _r
            _rembg_fn = _r
        except ImportError:
            import subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install",
                                   "rembg[cpu]", "-q", "--break-system-packages"])
            from rembg import remove as _r
            _rembg_fn = _r
        print("  [rembg] Loaded ✓")
    return _rembg_fn

def get_face_mask(img_tensor: torch.Tensor, threshold: float = 0.4) -> np.ndarray:
    H = W = img_tensor.shape[-1]
    try:
        rembg = get_rembg()
        arr   = ((img_tensor.clamp(-1,1) + 1) / 2 * 255).byte().permute(1,2,0).cpu().numpy()
        pil   = Image.fromarray(arr)
        rgba  = rembg(pil)
        alpha = np.array(rgba)[:, :, 3].astype(np.float32) / 255.0
        mask  = (alpha >= threshold).astype(np.float32)
        return mask
    except Exception as e:
        print(f"  [rembg] Failed ({e}) — ellipse fallback")
        cy, cx = H / 2.0, W / 2.0
        ry, rx = H * 0.48, W * 0.42
        ys = np.arange(H); xs = np.arange(W)
        YY, XX = np.meshgrid(ys, xs, indexing="ij")
        mask = (((YY - cy) / ry)**2 + ((XX - cx) / rx)**2 <= 1.0).astype(np.float32)
        return mask


# ─────────────────────────────────────────────────────────────
# SECTION 4 — FIXED SHADOW VISUALIZATIONS  [FIX-V2, FIX-V3]
# ─────────────────────────────────────────────────────────────

def compute_illumination_diff_map(img_src: torch.Tensor,
                                   img_tgt: torch.Tensor,
                                   mask: np.ndarray = None) -> np.ndarray:
    src = (img_src.float().clamp(-1, 1) + 1) / 2
    tgt = (img_tgt.float().clamp(-1, 1) + 1) / 2
    lum_src = (0.2126*src[0] + 0.7152*src[1] + 0.0722*src[2]).cpu().numpy()
    lum_tgt = (0.2126*tgt[0] + 0.7152*tgt[1] + 0.0722*tgt[2]).cpu().numpy()
    diff = lum_tgt - lum_src

    H, W = diff.shape
    pos = np.clip(diff, 0, 1)
    neg = np.clip(-diff, 0, 1)
    r = (255 * (1 - neg * 0.8)).astype(np.uint8)
    g = (255 * (1 - pos * 0.8 - neg * 0.8)).clip(0,255).astype(np.uint8)
    b = (255 * (1 - pos * 0.8)).astype(np.uint8)
    rgb = np.stack([r, g, b], axis=-1)

    if mask is not None:
        bg = mask < 0.5
        rgb[bg] = [80, 80, 80]

    return rgb.clip(0, 255).astype(np.uint8)

def compute_sh_face_projection(img: torch.Tensor,
                                sh_vec: np.ndarray,
                                mask: np.ndarray = None,
                                size: int = 256) -> np.ndarray:
    try:
        from scipy.ndimage import convolve
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "scipy", "-q", "--break-system-packages"])
        from scipy.ndimage import convolve

    sh_vec = np.clip(sh_vec, -10.0, 10.0)
    sh_r   = sh_vec[0:9]; sh_g = sh_vec[9:18]; sh_b = sh_vec[18:27]

    arr_np = (img.float().clamp(-1,1).cpu().numpy() + 1) / 2
    pil_img = Image.fromarray(
        (arr_np.transpose(1,2,0) * 255).clip(0,255).astype(np.uint8)
    ).resize((size, size), Image.LANCZOS)
    arr_np = np.array(pil_img).astype(np.float32) / 255.0
    lum = 0.2126*arr_np[:,:,0] + 0.7152*arr_np[:,:,1] + 0.0722*arr_np[:,:,2]

    H = W = size
    kx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32)
    ky = np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32)
    gx = convolve(lum, kx, mode='reflect')
    gy = convolve(lum, ky, mode='reflect')

    strength = 0.7
    nx_grad = -gx * strength
    ny_grad = -gy * strength

    xs = (np.arange(W) / W - 0.5) * 2
    ys = (np.arange(H) / H - 0.5) * 2
    XX, YY = np.meshgrid(xs, ys)
    r2 = np.clip(XX**2 + YY**2, 0, 1)
    nz_hemi = np.sqrt(np.clip(1.0 - r2 * 0.6, 0, 1))

    blend = 0.6
    nx = nx_grad * blend + XX * (1 - blend) * 0.3
    ny = ny_grad * blend + YY * (1 - blend) * 0.3
    nz = nz_hemi

    norm_len = np.sqrt(nx**2 + ny**2 + nz**2 + 1e-8)
    nx /= norm_len; ny /= norm_len; nz /= norm_len

    Y = np.stack([
        np.ones_like(nx)  * 0.282095,
        ny                * 0.488603,
        nz                * 0.488603,
        nx                * 0.488603,
        nx * ny           * 1.092548,
        ny * nz           * 1.092548,
        (3*nz*nz - 1)     * 0.315392,
        nx * nz           * 1.092548,
        (nx*nx - ny*ny)   * 0.546274,
    ], axis=-1)

    L_r = (Y * sh_r).sum(axis=-1)
    L_g = (Y * sh_g).sum(axis=-1)
    L_b = (Y * sh_b).sum(axis=-1)
    L   = np.stack([L_r, L_g, L_b], axis=-1)

    L_pos = np.clip(L, 0, None)
    mn    = L_pos.min(); mx = L_pos.max()
    L_norm = (L_pos - mn) / (mx - mn + 1e-8)
    L_mapped = np.power(L_norm, 0.4)
    out = (L_mapped * 255).clip(0, 255).astype(np.uint8)

    if mask is not None:
        mask_r = np.array(Image.fromarray(
            (mask * 255).astype(np.uint8)
        ).resize((size, size), Image.NEAREST)) / 255.0
        bg = mask_r < 0.5
        out[bg] = [20, 20, 20]

    # Light direction arrow
    sh_mean = (sh_r + sh_g + sh_b) / 3.0
    Lx, Ly, Lz = sh_mean[3], sh_mean[1], sh_mean[2]
    L_dir = np.array([Lx, Ly, Lz])
    L_norm_mag = np.linalg.norm(L_dir)
    if L_norm_mag > 0.05:
        L_dir /= L_norm_mag
        cx, cy = size // 2, size // 2
        scale  = size * 0.25
        ex = int(cx + L_dir[0] * scale)
        ey = int(cy - L_dir[1] * scale)
        pil_out = Image.fromarray(out)
        draw    = ImageDraw.Draw(pil_out)
        draw.line([(cx, cy), (ex, ey)], fill=(255, 255, 0), width=3)
        draw.ellipse([(cx-4, cy-4), (cx+4, cy+4)], fill=(255,255,0))
        out = np.array(pil_out)

    return out

def compute_pseudo_shadow_mask(img_src: torch.Tensor,
                                img_tgt: torch.Tensor,
                                mask: np.ndarray = None,
                                threshold: float = 0.08) -> np.ndarray:
    src = (img_src.float().clamp(-1,1) + 1) / 2
    tgt = (img_tgt.float().clamp(-1,1) + 1) / 2
    lum_src = (0.2126*src[0] + 0.7152*src[1] + 0.0722*src[2]).cpu().numpy()
    lum_tgt = (0.2126*tgt[0] + 0.7152*tgt[1] + 0.0722*tgt[2]).cpu().numpy()
    diff = lum_tgt - lum_src

    H, W = diff.shape
    mag  = np.abs(diff)

    shadow_col = np.array([220, 60, 60])
    lit_col    = np.array([60, 200, 80])
    bg_col     = np.array([50, 50, 60])
    base_col   = np.array([120, 120, 128])

    out = np.ones((H, W, 3), dtype=np.float32) * base_col

    shadow_m = diff < -threshold
    lit_m    = diff >  threshold
    intensity = np.clip(mag / 0.4, 0, 1)

    out[shadow_m] = (shadow_col * intensity[shadow_m, None] +
                     base_col * (1 - intensity[shadow_m, None]))
    out[lit_m]    = (lit_col * intensity[lit_m, None] +
                     base_col * (1 - intensity[lit_m, None]))

    if mask is not None:
        bg = mask < 0.5
        out[bg] = bg_col

    return out.clip(0, 255).astype(np.uint8)

def compute_abs_delta(img_src: torch.Tensor,
                       img_tgt: torch.Tensor,
                       mask: np.ndarray = None,
                       amplify: float = 4.0) -> np.ndarray:
    src = (img_src.float().clamp(-1,1) + 1) / 2
    tgt = (img_tgt.float().clamp(-1,1) + 1) / 2
    delta = (tgt - src).abs().clamp(0, 1) * amplify
    out = (delta.clamp(0,1).permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
    if mask is not None:
        bg = mask < 0.5
        out[bg] = [20, 20, 20]
    return out

def sh_to_sphere_shadow_map_colored(sh_vec: np.ndarray, size: int = 256) -> np.ndarray:
    sh_vec  = np.clip(sh_vec, -10.0, 10.0)
    sh_mean = (sh_vec[0:9] + sh_vec[9:18] + sh_vec[18:27]) / 3.0
    L = np.array([sh_mean[3], sh_mean[1], sh_mean[2]], dtype=np.float32)
    n = np.linalg.norm(L)
    if n > 1e-6: L /= n
    ys = (np.arange(size) + 0.5) / size * 2 - 1
    xs = (np.arange(size) + 0.5) / size * 2 - 1
    YY, XX = np.meshgrid(ys, xs, indexing="ij")
    r2 = XX**2 + YY**2; valid = r2 <= 1.0
    ZZ = np.where(valid, np.sqrt(np.clip(1.0 - r2, 0, 1)), 0.0)
    NdotL = np.clip(XX*L[0] + YY*L[1] + ZZ*L[2], 0, 1)
    NdotL[~valid] = 0.0
    r = (255 * np.clip(NdotL * 1.5, 0, 1)).astype(np.uint8)
    g = (255 * np.clip(NdotL * 1.5 - 0.5, 0, 1)).astype(np.uint8)
    b = (255 * np.clip(0.8 - NdotL * 1.2, 0, 1)).astype(np.uint8)
    rgb = np.stack([r, g, b], axis=-1)
    rgb[~valid] = [15, 15, 30]
    return rgb


# ─────────────────────────────────────────────────────────────
# SECTION 5 — MASKED LOSS DIAGNOSTICS  [FIX-V4]
# ─────────────────────────────────────────────────────────────

def check_masked_loss(batch: dict, masks: dict = None, device: str = "cpu") -> dict:
    print(f"\n{'='*60}")
    print("  MASKED LOSS DIAGNOSTICS  [FIX-V4]")
    print(f"{'='*60}")

    img_src = batch["img_src"].to(device)
    img_tgt = batch["img_tgt"].to(device)
    B = img_src.shape[0]

    l1_all = F.l1_loss(img_src, img_tgt).item()
    print(f"\n  [Unmasked L1]   {l1_all:.4f}")
    print(f"    (your v1 result was 0.5005 — suspiciously high for same-angle pairs)")

    print(f"\n  Per-sample luminance diff:")
    for i in range(B):
        src = (img_src[i].clamp(-1,1) + 1) / 2
        tgt = (img_tgt[i].clamp(-1,1) + 1) / 2
        diff = (tgt - src).abs()
        lum  = 0.2126*diff[0] + 0.7152*diff[1] + 0.0722*diff[2]
        print(f"    Sample {i}: mean={lum.mean():.4f}  max={lum.max():.4f}"
              f"  src={batch['src_hdri'][i][:25]}"
              f"  tgt={batch['tgt_hdri'][i][:25]}")

    print(f"\n  Computing foreground masks (rembg)...")
    fg_l1s = []; bg_l1s = []
    for i in range(min(4, B)):
        mask = get_face_mask(img_src[i].cpu(), threshold=0.4)
        fg   = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0).to(device)
        bg   = 1.0 - fg
        n_fg = fg.sum().item(); n_bg = bg.sum().item()
        diff = (img_src[i:i+1] - img_tgt[i:i+1]).abs()
        l1_fg = (diff * fg).sum().item() / (n_fg * 3 + 1e-8)
        l1_bg = (diff * bg).sum().item() / (n_bg * 3 + 1e-8)
        fg_l1s.append(l1_fg); bg_l1s.append(l1_bg)
        print(f"    Sample {i}: FG mask={n_fg/(256*256)*100:.1f}%  "
              f"L1_face={l1_fg:.4f}  L1_bg={l1_bg:.4f}  "
              f"ratio={l1_bg/(l1_fg+1e-8):.1f}×")

    print(f"\n  Summary:")
    print(f"    Mean FG (face) L1   : {np.mean(fg_l1s):.4f}")
    print(f"    Mean BG (scene) L1  : {np.mean(bg_l1s):.4f}")
    if np.mean(fg_l1s) > 0.01:
        ratio = np.mean(bg_l1s) / np.mean(fg_l1s)
        print(f"    BG/FG ratio         : {ratio:.1f}×")
        if ratio > 2.0:
            print(f"    !! Background dominates loss by {ratio:.1f}× !!")
            print(f"       This confirms rembg mask is essential in training.")
        else:
            print(f"    ✓ Face and background contribute similarly to loss")

    return {"l1_all": l1_all, "l1_face": np.mean(fg_l1s), "l1_bg": np.mean(bg_l1s)}


# ─────────────────────────────────────────────────────────────
# SECTION 6 — DIAGNOSTIC GRID (v2, with masks)
# ─────────────────────────────────────────────────────────────

def _tensor_to_pil(t):
    t = t.squeeze().clamp(-1,1)
    return Image.fromarray(((t+1)/2*255).byte().permute(1,2,0).cpu().numpy())

def _np_to_pil(arr): return Image.fromarray(arr.clip(0,255).astype(np.uint8))

def _label(img, text, bg=(20,20,20)):
    W, H   = img.size
    canvas = Image.new("RGB", (W, H+22), bg)
    canvas.paste(img, (0, 22))
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 12)
    except Exception:
        font = ImageFont.load_default()
    draw.text((3, 3), text, fill=(220,220,220), font=font)
    return canvas

def build_diagnostic_grid_v2(batch: dict, n_samples: int = 4,
                               save_path: Path = None,
                               compute_masks: bool = True) -> Image.Image:
    n_samples = min(n_samples, batch["img_src"].shape[0])
    S    = batch["img_src"].shape[-1]
    CW   = S; CH = S + 22
    COLS = 7; PAD = 5; HDR = 55

    GW = COLS * CW + (COLS+1)*PAD
    GH = n_samples * CH + (n_samples+1)*PAD + HDR
    grid = Image.new("RGB", (GW, GH), (12, 12, 20))
    draw = ImageDraw.Draw(grid)

    try:
        fh = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 15)
        fs = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 11)
    except Exception:
        fh = fs = ImageFont.load_default()

    draw.text((PAD, 5),
              "RELIGHT DIAGNOSTIC v2 — Masked Face Shadow Viz | Same-Angle Pairs",
              fill=(255,210,50), font=fh)

    col_headers = [
        "Source (HDRI A)",
        "Target (HDRI B) ✓SAME ANGLE",
        "Illum Diff [MASKED]",
        "Shadow Mask [MASKED]",
        "SH→Face [FIXED normals]",
        "OLD Sphere [DEPRECATED]",
        "Abs Delta ×4 [MASKED]",
    ]
    for ci, label in enumerate(col_headers):
        draw.text((PAD + ci*(CW+PAD) + 3, 28), label, fill=(130,180,255), font=fs)

    for ri in range(n_samples):
        src    = batch["img_src"][ri]
        tgt    = batch["img_tgt"][ri]
        sh_tgt = batch["sh_tgt"][ri].numpy()
        yaw    = batch["camera_yaw"][ri].item()
        s_hdri = batch["src_hdri"][ri][:22]
        t_hdri = batch["tgt_hdri"][ri][:22]
        is_id  = batch["is_identity"][ri].item()

        s_lum = (0.2126*(src[0]+1)/2 + 0.7152*(src[1]+1)/2 + 0.0722*(src[2]+1)/2).mean().item()
        t_lum = (0.2126*(tgt[0]+1)/2 + 0.7152*(tgt[1]+1)/2 + 0.0722*(tgt[2]+1)/2).mean().item()
        lum_diff = abs(t_lum - s_lum)

        mask = None
        if compute_masks:
            try:
                mask = get_face_mask(src, threshold=0.4)
            except Exception:
                pass

        illum  = compute_illumination_diff_map(src, tgt, mask=mask)
        shadow = compute_pseudo_shadow_mask(src, tgt, mask=mask, threshold=0.08)
        sh_face = compute_sh_face_projection(tgt, sh_tgt, mask=mask, size=S)
        sphere  = sh_to_sphere_shadow_map_colored(sh_tgt, size=S)
        delta   = compute_abs_delta(src, tgt, mask=mask, amplify=4.0)

        id_tag = " [IDENTITY]" if is_id else f" Δlum={lum_diff:.3f}"
        panels = [
            _label(_tensor_to_pil(src),   f"SRC yaw={yaw:.0f}° {s_hdri}"),
            _label(_tensor_to_pil(tgt),   f"TGT yaw={yaw:.0f}° {t_hdri}{id_tag}"),
            _label(_np_to_pil(illum),     "R=bright B=dark [fg masked]"),
            _label(_np_to_pil(shadow),    "R=shadow G=lit [fg masked]"),
            _label(_np_to_pil(sh_face),   "SH on face normals + arrow"),
            _label(_np_to_pil(sphere),    "OLD sphere [misaligned]"),
            _label(_np_to_pil(delta),     "Abs(tgt-src)×4 [fg masked]"),
        ]

        y_off = HDR + ri * (CH + PAD) + PAD
        for ci, panel in enumerate(panels):
            x = PAD + ci * (CW + PAD)
            p = panel.convert("RGB").resize((CW, CH), Image.LANCZOS)
            grid.paste(p, (x, y_off))

    if save_path:
        grid.save(str(save_path))
        print(f"  [SAVED] {save_path}")
    return grid


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

def run_all_diagnostics_v2():
    print("\n" + "="*60)
    print("  RELIGHT PIPELINE — DIAGNOSTIC v2 (FIXED & CORRECTED)")
    print("="*60)

    # 1. Checkpoint
    last_ckpt = CKPT_DIR / "last.pt"
    if last_ckpt.exists():
        verify_checkpoint(last_ckpt)
    else:
        print(f"  [INFO] No checkpoint at {last_ckpt}")

    # 2. CSV structure
    csv_findings = inspect_csv_structure(CSV_PATH)

    # 3. Dataset
    sh_npy  = SH_CACHE_DIR / "sh_coeffs.npy"
    sh_json = SH_CACHE_DIR / "sh_index.json"

    if not sh_npy.exists():
        print("\n  [SKIP] SH files missing — run precompute_sh_if_needed() first")
        return

    print(f"\n{'='*60}")
    print("  LOADING DATASET V2")
    print(f"{'='*60}")

    ds = RelightDatasetV2(
        csv_path      = str(CSV_PATH),
        sh_npy_path   = str(sh_npy),
        sh_index_path = str(sh_json),
        image_size    = 256,
        split         = "train",
        val_fraction  = 0.02,
        # force_yaw_col = "camera_yaw_deg",  # uncomment if inspector suggests correct column
    )

    loader = DataLoader(ds, batch_size=6, shuffle=True, num_workers=0)
    batch  = next(iter(loader))

    # 4. Masked loss diagnostics
    loss_stats = check_masked_loss(batch, device=DEVICE)

    # 5. Diagnostic grid v2
    print(f"\n{'='*60}")
    print("  BUILDING DIAGNOSTIC GRID v2")
    print(f"{'='*60}")
    grid_path = VIZ_DIR / "diagnostic_grid_v2.png"
    grid = build_diagnostic_grid_v2(
        batch, n_samples=4, save_path=grid_path, compute_masks=True
    )

    # 6. Summary
    print(f"\n{'='*60}")
    print("  SUMMARY")
    print(f"{'='*60}")
    print(f"  Angle groups  : {len(ds.angle_groups)}")
    print(f"    → Expected ~{45 * 4} if yaw column is correct")
    print(f"    → If still 3, the JSON has no per-image yaw field")
    print(f"       Check CSV inspector output above for the raw JSON keys")
    print(f"  L1 unmasked   : {loss_stats['l1_all']:.4f}")
    print(f"  L1 face only  : {loss_stats['l1_face']:.4f}")
    print(f"  L1 bg only    : {loss_stats['l1_bg']:.4f}")
    print(f"\n  OUTPUT: {VIZ_DIR}/diagnostic_grid_v2.png")
    print(f"{'='*60}\n")

    return {"csv": csv_findings, "loss": loss_stats, "n_groups": len(ds.angle_groups)}

# Run
if __name__ == "__main__":
    results = run_all_diagnostics_v2()

In [ ]:
import json
import sys
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# 1. Smart Path Finder
def find_file(filename):
    for base in [Path("/kaggle/working"), Path("/kaggle/input")]:
        found = list(base.rglob(filename))
        if found: return found[0]
    raise FileNotFoundError(f"Could not find {filename}.")

try:
    CSV_PATH = find_file("master_metadata.csv")
    SH_NPY_PATH = find_file("sh_coeffs.npy")
    SH_JSON_PATH = find_file("sh_index.json")
except FileNotFoundError as e:
    print(f"\n❌ ERROR: {e}")
    print("FIX: Run your main training cell for 30 seconds, hit STOP, then run this again.")
    sys.exit()

# 2. Standalone Shadow Physics Logic
def sh_to_shadow_map(sh_vec, size=64):
    sh_vec  = np.clip(sh_vec, -10.0, 10.0)
    sh_r    = sh_vec[0:9]; sh_g = sh_vec[9:18]; sh_b = sh_vec[18:27]
    sh_mean = (sh_r + sh_g + sh_b) / 3.0
    Ly = sh_mean[1]; Lz = sh_mean[2]; Lx = sh_mean[3]
    L  = np.array([Lx, Ly, Lz], dtype=np.float32)
    norm = np.linalg.norm(L)
    if norm < 1e-6:
        return np.ones((size, size), dtype=np.float32) * 0.5
    L = L / norm
    ys = (np.arange(size) + 0.5) / size * 2 - 1
    xs = (np.arange(size) + 0.5) / size * 2 - 1
    YY, XX = np.meshgrid(ys, xs, indexing="ij")
    r2    = XX**2 + YY**2
    valid = r2 <= 1.0
    ZZ    = np.where(valid, np.sqrt(np.clip(1.0 - r2, 0, 1)), 0.0)
    NdotL = XX * L[0] + YY * L[1] + ZZ * L[2]
    shadow_map = np.clip(NdotL, 0.0, 1.0).astype(np.float32)
    shadow_map[~valid] = 0.0
    return shadow_map

def _normalise_hdri_key(raw):
    key = raw.strip().lower()
    for ext in (".hdr", ".exr", ".png"):
        if key.endswith(ext):
            key = key[:-len(ext)]; break
    return key

# 3. Load Data & Generate Plots
print("\nLoading physical data into memory...")
df = pd.read_csv(CSV_PATH)
sh_coeffs = np.load(SH_NPY_PATH)
with open(SH_JSON_PATH) as f:
    sh_index = {_normalise_hdri_key(k): v for k, v in json.load(f).items()}

# Pick 3 random images to visualize
samples = df.sample(3)

fig, axes = plt.subplots(3, 6, figsize=(20, 10.5))
titles = [
    "1. GT Image", 
    "2. Raw Shadow Map\n(N·L Projection)", 
    "3. Bright Mask\n(Forces > 0.7)", 
    "4. Dark Mask\n(Forces < 0.2)", 
    "5. Boundary Mask\n(0.3 to 0.7)", 
    "6. Color Overlay\n(Red=Bright, Blue=Dark)"
]
for j, title in enumerate(titles):
    axes[0, j].set_title(title, fontsize=12, fontweight='bold')

for i, (_, row) in enumerate(samples.iterrows()):
    img_path = row["image_path"]
    try:
        img = Image.open(img_path).convert("RGB").resize((256, 256))
        img_np = np.array(img) / 255.0
    except Exception as e:
        continue
    
    # --- BUG FIX: ROBUST DICTIONARY LOOKUP ---
    norm_key = _normalise_hdri_key(str(row["hdri_name_key"]))
    if norm_key in sh_index:
        idx = sh_index[norm_key]
        print(f"Row {i+1}: Exact match for HDRI '{norm_key}'")
    else:
        # Fuzzy match to fix silent fallback bug
        matches = difflib.get_close_matches(norm_key, sh_index.keys(), n=1, cutoff=0.2)
        if matches:
            idx = sh_index[matches[0]]
            print(f"Row {i+1}: Fuzzy matched '{norm_key}' -> '{matches[0]}'")
        else:
            idx = 0
            print(f"Row {i+1}: ❌ FAILED to match '{norm_key}'. Defaulting to Index 0.")

    sh2 = sh_coeffs[idx]
    s_map = sh_to_shadow_map(sh2, size=64)
    
    b_mask = (s_map > 0.7).astype(float)
    d_mask = (s_map < 0.2).astype(float)
    bound_mask = ((s_map > 0.3) & (s_map < 0.7)).astype(float)
    
    # --- VISUAL UPGRADE FOR COLUMN 2 ---
    axes[i, 0].imshow(img_np); axes[i, 0].axis('off')
    
    axes[i, 1].imshow(s_map, cmap='gray', vmin=0, vmax=1)
    axes[i, 1].contour(s_map, levels=8, colors='cyan', linewidths=0.8, alpha=0.7)
    axes[i, 1].axis('off')
    # -----------------------------------
    
    axes[i, 2].imshow(b_mask, cmap='gray', vmin=0, vmax=1); axes[i, 2].axis('off')
    axes[i, 3].imshow(d_mask, cmap='gray', vmin=0, vmax=1); axes[i, 3].axis('off')
    axes[i, 4].imshow(bound_mask, cmap='gray', vmin=0, vmax=1); axes[i, 4].axis('off')
    
    overlay = np.zeros((64, 64, 3))
    overlay[b_mask == 1] = [1, 0, 0]      
    overlay[d_mask == 1] = [0, 0.3, 1]    
    overlay[bound_mask == 1] = [0, 1, 0]  
    axes[i, 5].imshow(s_map, cmap='gray', alpha=0.4)
    axes[i, 5].imshow(overlay, alpha=0.7)
    axes[i, 5].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
!zip -r important_files.zip  /kaggle/working/best.pt /kaggle/working/last.pt

In [ ]:
from IPython.display import FileLink
FileLink('important_files.zip')

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   FACE RELIGHTING — COMPLETE INFERENCE CELL  (Final Corrected Version)      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Camera geometry re-derived from render_faces2.py:                          ║
║    CAMERA_LENS_MM=50, sensor=36mm → FOV=39.5978°                           ║
║    CAMERA_POSITIONS={'front':-45} → orbit at -45° → LOOKS at +45° azimuth  ║
║    CAMERA_ELEVATION=0.08 → actual look elevation = 2.94° (not 4.57°)       ║
║    rotate_hdri(deg) → Mapping Z CCW → sample at (45° - hdri_rotation_deg)  ║
║    Blender Filmic → approximated with ACES curve (better than Reinhard)     ║
║                                                                              ║
║  PART A — Quantitative Metrics: MSE · PSNR · SSIM · LPIPS                  ║
║  PART B — Visual Comparison Grid: Input | GT | Predicted + ZIP              ║
║  PART C — Lighting Demo HTML slider with HDRI background compositing        ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ── CONFIG ────────────────────────────────────────────────────────────────────
EVAL_CFG = {
    "n_eval"       : 60,
    "ddim_steps"   : 150,
    "batch_size"   : 4,
    "samples_per_grid": 32,   # <-- add this
    "n_grids"         : 20,   # <-- add this
    "image_size"   : 256,
    "demo_n_hdri"  : 20,
    "demo_row"     : 0,
    "grid_cols"    : 4,
    "val_fraction" : 0.02,
    "val_seed"     : 42,
    "show_bg_demo" : True,
}

import os, sys, json, math, random, time, shutil, warnings, subprocess, zipfile
import numpy as np, base64, io
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}  |  PyTorch: {torch.__version__}")

def _pip(pkg, import_as=None):
    try: __import__(import_as or pkg)
    except ImportError:
        print(f"  Installing {pkg}...")
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])

_pip("scikit-image","skimage"); _pip("lpips"); _pip("rembg[cpu]","rembg")

from skimage.metrics import structural_similarity as _ssim_fn
from lpips import LPIPS as _LPIPS

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT    = Path("/kaggle/input")
WORK     = Path("/kaggle/working")
EVAL_DIR = WORK / "eval_outputs"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# ── Find best checkpoint ──────────────────────────────────────────────────────
def _valid_pt(p):
    try:
        ck = torch.load(p, map_location="cpu")
        return True, ck.get("step",0), ck.get("best_loss",float("inf"))
    except: return False, 0, float("inf")

best_path, best_loss = None, float("inf")
print("\n  Scanning checkpoints...")
for ds in ["relight-best","relight-checkpoint"]:
    for root in INPUT.rglob(ds):
        if not root.is_dir(): continue
        for pt in sorted(root.glob("*.pt")):
            ok,step,loss = _valid_pt(pt)
            flag = ""
            if ok and loss < best_loss:
                best_loss=loss; best_path=pt; flag="  <- selected"
            print(f"    {pt.name:<22} {'step='+str(step)+' loss='+f'{loss:.4f}' if ok else 'CORRUPT'}{flag}")

if best_path is None:
    raise FileNotFoundError("No valid checkpoint found.")
ckpt      = torch.load(best_path, map_location=DEVICE)
ckpt_step = ckpt.get("step","?")
print(f"\n  Using: {best_path.name}  step={ckpt_step}  loss={best_loss:.4f}")

# ── HDRI cache ────────────────────────────────────────────────────────────────
HDRI_CACHE = next((p for p in INPUT.rglob("hdri_cache") if p.is_dir()), None)
if HDRI_CACHE is None: raise FileNotFoundError("hdri_cache not found")
print(f"  HDRI cache : {HDRI_CACHE}")

# ── Metadata CSV ──────────────────────────────────────────────────────────────
CSV_PATH = next((p for p in INPUT.rglob("master_metadata.csv")), None)
if CSV_PATH is None: raise FileNotFoundError("master_metadata.csv not found")
import pandas as pd
df_meta = pd.read_csv(str(CSV_PATH))
print(f"  Metadata   : {len(df_meta)} rows")

# ── SH cache ─────────────────────────────────────────────────────────────────
# Check multiple locations — working dir, input datasets
SH_DIR  = WORK/"sh_cache"
SH_NPY  = SH_DIR/"sh_coeffs.npy"
SH_JSON = SH_DIR/"sh_index.json"

# Also check if sh_cache uploaded as a dataset
if not SH_NPY.exists():
    for _p in INPUT.rglob("sh_coeffs.npy"):
        SH_NPY  = _p
        SH_JSON = _p.parent/"sh_index.json"
        SH_DIR  = _p.parent
        print(f"  SH cache found in dataset: {SH_DIR}")
        break

# If still not found — precompute from hdri_cache (takes ~2 min)
if not SH_NPY.exists():
    print("  SH cache not found — precomputing from HDRI cache (~2 min)...")
    SH_DIR.mkdir(parents=True, exist_ok=True)

    def _build_sh_dirs(H, W):
        phi   = (np.arange(W)+0.5)/W*2*np.pi
        theta = (np.arange(H)+0.5)/H*np.pi
        THETA, PHI = np.meshgrid(theta, phi, indexing='ij')
        st=np.sin(THETA); ct=np.cos(THETA)
        sp=np.sin(PHI);   cp=np.cos(PHI)
        x=st*cp; y=st*sp; z=ct
        Y=np.stack([0.282095*np.ones_like(x),
                    0.488603*y,0.488603*z,0.488603*x,
                    1.092548*x*y,1.092548*y*z,
                    0.315392*(3*z*z-1),1.092548*x*z,
                    0.546274*(x*x-y*y)],axis=-1)
        return Y, st*(np.pi/H)*(2*np.pi/W)

    def _compute_sh(hdr_npy):
        img=((hdr_npy.astype(np.float64)+1.0)/2.0).clip(0)
        C,H,W=img.shape; Y,sa=_build_sh_dirs(H,W)
        return np.concatenate([(img[c][:,:,None]*Y*sa[:,:,None]).sum(axis=(0,1))
                                for c in range(C)]).astype(np.float32)

    hdr_files = sorted(HDRI_CACHE.glob("*_hdr.npy"))
    all_coeffs=[]; sh_idx={}
    for i,fp in enumerate(hdr_files):
        arr=np.load(fp)
        if arr.ndim!=3 or arr.shape[0]!=3: continue
        key=fp.stem.replace("_hdr","").strip().lower()
        sh_idx[key]=len(all_coeffs)
        all_coeffs.append(_compute_sh(arr))
        if (i+1)%100==0: print(f"    [{i+1}/{len(hdr_files)}]")
    arr_out=np.stack(all_coeffs)
    np.save(SH_NPY, arr_out)
    with open(SH_JSON,"w") as f: json.dump(sh_idx, f)
    print(f"  SH precomputed: {arr_out.shape} ✓")

sh_coeffs_all = np.load(SH_NPY)
with open(SH_JSON) as f: _sh_raw = json.load(f)

def _nk(k):
    k = k.strip().lower()
    for e in (".hdr",".exr",".png"): k=k[:-len(e)] if k.endswith(e) else k
    return k

sh_index = {_nk(k):v for k,v in _sh_raw.items()}
print(f"  SH cache   : {sh_coeffs_all.shape[0]} HDRIs")

# =============================================================================
# MODEL DEFINITIONS -- exact match to kaggle_train_v3.py
# =============================================================================

class AdaGN(nn.Module):
    def __init__(self,nc,ze=256,ng=32):
        super().__init__()
        while nc%ng!=0 and ng>1: ng//=2
        self.norm=nn.GroupNorm(ng,nc,affine=False)
        self.proj=nn.Linear(ze,nc*2)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)
    def forward(self,x,ze):
        h=self.norm(x); s,b=self.proj(ze).chunk(2,dim=-1)
        return h*(s[:,:,None,None]+1)+b[:,:,None,None]

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, ze_dim=256):
        super().__init__()
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.adagn1 = AdaGN(out_ch, ze_dim)
        self.adagn2 = AdaGN(out_ch, ze_dim)
        self.act    = nn.SiLU()
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, cond):
        h = self.act(self.adagn1(self.conv1(x), cond))
        h = self.act(self.adagn2(self.conv2(h), cond))
        return h + self.skip(x)

class Downsample(nn.Module):
    def __init__(self, ch): super().__init__(); self.c=nn.Conv2d(ch,ch,3,stride=2,padding=1)
    def forward(self, x): return self.c(x)

class Upsample(nn.Module):
    def __init__(self, ch): super().__init__(); self.c=nn.Conv2d(ch,ch,3,padding=1)
    def forward(self, x): return self.c(F.interpolate(x,scale_factor=2,mode="nearest"))

class SinPosEmb(nn.Module):
    def __init__(self,d): super().__init__(); self.d=d
    def forward(self,t):
        h=self.d//2
        f=torch.exp(-math.log(10000)*torch.arange(h,device=t.device)/(h-1))
        a=t[:,None].float()*f[None]
        return torch.cat([a.sin(),a.cos()],dim=-1)

class TEmb(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.sin = SinPosEmb(d)
        self.mlp = nn.Sequential(nn.Linear(d, d*4), nn.SiLU(), nn.Linear(d*4, d))
    def forward(self, t): return self.mlp(self.sin(t))

class UNet(nn.Module):
    def __init__(self, in_channels=8, base_ch=128, ch_mult=(1,2,2,2), ze_dim=256):
        super().__init__()
        self.t_emb   = TEmb(ze_dim)
        chs          = [base_ch * m for m in ch_mult]
        self.in_proj = nn.Conv2d(in_channels, chs[0], 3, padding=1)
        self.enc_blocks  = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        prev = chs[0]
        for ch in chs[:-1]:
            self.enc_blocks.append(ResBlock(prev, ch, ze_dim))
            self.downsamples.append(Downsample(ch))
            prev = ch
        self.mid1 = ResBlock(prev, chs[-1], ze_dim)
        self.mid2 = ResBlock(chs[-1], chs[-1], ze_dim)
        prev = chs[-1]
        self.upsamples  = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        for ch in reversed(chs[:-1]):
            self.upsamples.append(Upsample(prev))
            self.dec_blocks.append(ResBlock(prev + ch, ch, ze_dim))
            prev = ch
        self.out_norm = nn.GroupNorm(32, prev)
        self.out_proj = nn.Conv2d(prev, 4, 3, padding=1)
        nn.init.zeros_(self.out_proj.weight); nn.init.zeros_(self.out_proj.bias)

    def forward(self, x, t, ze):
        cond = ze + self.t_emb(t)
        h    = self.in_proj(x)
        skips = []
        for blk, down in zip(self.enc_blocks, self.downsamples):
            h = blk(h, cond); skips.append(h); h = down(h)
        h = self.mid2(self.mid1(h, cond), cond)
        for up, blk, sk in zip(self.upsamples, self.dec_blocks, reversed(skips)):
            h = up(h); h = torch.cat([h, sk], dim=1); h = blk(h, cond)
        return self.out_proj(F.silu(self.out_norm(h)))

class LP3(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(36,256),nn.ReLU(True),
            nn.Linear(256,512),nn.ReLU(True),
            nn.Linear(512,256))
    def forward(self,x): return self.net(x)

class DDPM:
    def __init__(self,T=1000,device="cuda"):
        self.T=T
        t=torch.linspace(0,T,T+1,device=device)
        f=torch.cos(((t/T)+0.008)/1.008*math.pi/2)**2; f=f/f[0]
        ab=(f[1:]/f[:-1]).cumprod(0).clamp(1e-5,1-1e-5)
        self.alpha_bar=ab; self.sqrt_ab=ab.sqrt(); self.sqrt_one_m_ab=(1-ab).sqrt()
    def q_sample(self,x0,t,noise=None):
        if noise is None: noise=torch.randn_like(x0)
        return self.sqrt_ab[t][:,None,None,None]*x0+self.sqrt_one_m_ab[t][:,None,None,None]*noise,noise

# ── Load models ───────────────────────────────────────────────────────────────
print("\n  Loading VAE...")
from diffusers import AutoencoderKL
vae=AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(DEVICE)
vae.eval()
for p in vae.parameters(): p.requires_grad_(False)

# Detect base_ch from checkpoint — in_proj outputs base_ch channels
_w = (ckpt["unet"].get("in_proj.weight") or
      ckpt["unet"].get("_orig_mod.in_proj.weight"))
base_ch = _w.shape[0]  # (base_ch, 8, 3, 3) -> shape[0] = base_ch

lp3=LP3().to(DEVICE); unet=UNet(in_channels=8,base_ch=base_ch).to(DEVICE)
unet.load_state_dict({k.replace("_orig_mod.",""):v for k,v in ckpt["unet"].items()}, strict=False)
try: lp3.load_state_dict(ckpt["lp3"]); print("  LP3 loaded OK")
except RuntimeError: print("  LP3 shape mismatch -- random init (evaluation still valid for UNet)")

unet.eval(); lp3.eval()
ddpm=DDPM(T=1000,device=DEVICE)

print("  Loading LPIPS...")
lpips_fn=_LPIPS(net="vgg").to(DEVICE); lpips_fn.eval()
for p in lpips_fn.parameters(): p.requires_grad_(False)
print(f"  Params: {sum(p.numel() for p in list(lp3.parameters())+list(unet.parameters()))/1e6:.1f}M  base_ch={base_ch}")
print("  All models ready")

@torch.no_grad()
def vae_encode(x): return vae.encode(x).latent_dist.sample()*0.18215
@torch.no_grad()
def vae_decode(z): return vae.decode(z/0.18215).sample.clamp(-1,1)

@torch.no_grad()
def ddim_sample(z1,ze_inp,n_steps):
    ze=lp3(ze_inp)
    ts=torch.linspace(ddpm.T-1,0,n_steps,dtype=torch.long).tolist()
    x=torch.randn_like(z1)
    for i,t in enumerate(ts):
        t=int(t); tb=torch.full((z1.shape[0],),t,device=DEVICE,dtype=torch.long)
        eps=unet(torch.cat([x,z1],dim=1),tb,ze)
        ab=ddpm.alpha_bar[t]; x0=((x-(1-ab).sqrt()*eps)/ab.sqrt()).clamp(-1,1)
        if i<len(ts)-1:
            abp=ddpm.alpha_bar[int(ts[i+1])]; x=abp.sqrt()*x0+(1-abp).sqrt()*eps
        else: x=x0
    return x

# =============================================================================
# REMBG MASKS -- one per subject, ~2 seconds total
# =============================================================================

def build_subject_masks(size=256):
    from rembg import remove as _rembg
    masks={}; sids=sorted(df_meta["subject_id"].unique())
    print(f"\n  Computing rembg masks for {len(sids)} subjects...")
    for sid in sids:
        sid=int(sid); rows=df_meta[df_meta["subject_id"]==sid]
        if not len(rows): continue
        try:
            pil=Image.open(rows.iloc[0]["image_path"]).convert("RGB").resize((size,size))
            alpha=(np.array(_rembg(pil))[:,:,3]/255.0>=0.5).astype(np.float32)
            masks[sid]=torch.from_numpy(alpha).unsqueeze(0).unsqueeze(0).to(DEVICE)
            print(f"    Subject {sid}: OK ({alpha.mean()*100:.1f}% fg)")
        except Exception as e:
            print(f"    Subject {sid}: rembg failed ({e}) -- ellipse fallback")
            H=W=size; cy,cx=H/2,W/2; ry,rx=H*.45,W*.40
            ys=torch.arange(H,device=DEVICE).float(); xs=torch.arange(W,device=DEVICE).float()
            yy,xx=torch.meshgrid(ys,xs,indexing="ij")
            masks[sid]=(((yy-cy)/ry)**2+((xx-cx)/rx)**2<=1).float().unsqueeze(0).unsqueeze(0)
    print("  Masks ready"); return masks

SUBJECT_MASKS=build_subject_masks(EVAL_CFG["image_size"])

def apply_mask(img,sid):
    m=SUBJECT_MASKS.get(int(sid))
    return img*m if m is not None else img

# =============================================================================
# HDRI BACKGROUND COMPOSITOR
# Re-derived carefully from render_faces2.py source code.
#
# Two key corrections from previous version:
#
# CORRECTION 1 -- Camera look azimuth = +45deg (NOT -45deg)
#   CAMERA_POSITIONS={'front': -45} is the ORBIT angle.
#   Camera sits at XY = (sin(-45)*d, -cos(-45)*d) = (-0.707d, -0.707d)
#   Camera LOOKS toward face at origin:
#     look_dir = (0,0,0.42) - (-0.707d, -0.707d, 0.08d) = (+0.707d, +0.707d, ...)
#     azimuth = atan2(+0.707, +0.707) = +45deg in Blender XY convention
#
# CORRECTION 2 -- Camera look elevation = 2.94deg (NOT 4.57deg)
#   Previous used arctan(CAMERA_ELEVATION=0.08) = 4.57deg
#   Actual: camera at Z=0.08*dist=0.256, aim_target at Z=0.42
#   Look dir Z component = (0.42-0.256)/norm = 0.164/3.26 = 0.0513
#   Elevation = arcsin(0.0513) = 2.94deg
#
# CORRECTION 3 -- HDRI rotation sign
#   rotate_hdri(deg) sets Mapping node Z rotation = radians(deg) (CCW)
#   CCW rotation of panorama: content at azimuth A moves to azimuth A+deg
#   Camera at azimuth 45 sees content originally at: 45 - hdri_rotation_deg
#
# CORRECTION 4 -- Filmic tonemap (not Reinhard)
#   Blender uses Filmic view transform, approximated here with ACES curve
# =============================================================================

# Verified constants from render_faces2.py:
_FOV_DEG      = 2*math.degrees(math.atan(36.0/(2*50.0)))  # 39.5978 degrees
_CAM_LOOK_AZ  = 45.0     # camera LOOKS at +45 degrees azimuth (orbit=-45 -> look=+45)
_CAM_LOOK_EL  = 2.9387   # camera look elevation in degrees (verified by geometry)


def _aces_filmic(x):
    """
    ACES fitted curve approximation (Krzysztof Narkowicz).
    Better approximation of Blender Filmic than Reinhard.
    x: linear HDR numpy array (any shape, non-negative)
    returns: tonemapped [0,1] with gamma 2.2
    """
    x = x.clip(0)
    a,b,c,d,e = 2.51, 0.03, 2.43, 0.59, 0.14
    t = (x*(a*x+b)) / (x*(c*x+d)+e)
    return np.power(t.clip(0,1), 1.0/2.2)


def hdri_to_background(hdr_npy, hdri_rotation_deg,
                        camera_yaw_deg=-45.0,
                        exposure_ev=0.0, out_size=256):
    """
    Projects HDRI panorama to exact perspective background matching Blender render.

    Parameters
    ----------
    hdr_npy           : (3,H,W) numpy -- *_hdr.npy from cache, stored in [-1,1]
    hdri_rotation_deg : from metadata["hdri_rotation_deg"]
    camera_yaw_deg    : from metadata["camera_yaw_deg"] (always -45 for front)
    exposure_ev       : from metadata["exposure_used"]
    out_size          : 256 to match renders

    Returns
    -------
    (out_size, out_size, 3) float32 in [0,1] -- tonemapped, gamma-corrected
    """
    # Decode: stored as [-1,1] normalised mean-subtracted
    hdr = ((hdr_npy.astype(np.float32)+1.0)/2.0).clip(0)
    _,H,W = hdr.shape
    hdr = hdr.transpose(1,2,0)  # (H,W,3) equirectangular

    # Camera look direction (corrected geometry)
    # Orbit angle -> look azimuth: orbit=-45 -> look=+45 degrees
    # formula: look_az = -(orbit_angle) when camera orbits around origin
    look_az_deg  = _CAM_LOOK_AZ   # 45.0
    look_el_deg  = _CAM_LOOK_EL   # 2.94

    # HDRI sampling azimuth: camera at +45 sees what was originally at 45-hdri_rot
    sample_az_deg = look_az_deg - hdri_rotation_deg
    az_rad = np.deg2rad(sample_az_deg)
    el_rad = np.deg2rad(look_el_deg)

    # Build perspective ray grid
    focal = (out_size/2.0) / np.tan(np.deg2rad(_FOV_DEG/2.0))
    xs = np.arange(out_size)-out_size/2.0+0.5
    ys = np.arange(out_size)-out_size/2.0+0.5
    XX,YY = np.meshgrid(xs,ys)

    # Camera-space rays: camera looks along +Z, X=right, Y=up
    Xc=XX/focal; Yc=-YY/focal; Zc=np.ones_like(XX)
    n=np.sqrt(Xc**2+Yc**2+Zc**2); Xc/=n; Yc/=n; Zc/=n

    # Apply pitch (elevation tilt -- camera looks slightly upward)
    cp=np.cos(el_rad); sp=np.sin(el_rad)
    Xp=Xc; Yp=cp*Yc-sp*Zc; Zp=sp*Yc+cp*Zc

    # Apply yaw (horizontal rotation to face the correct panorama azimuth)
    ca=np.cos(az_rad); sa=np.sin(az_rad)
    Xw=ca*Xp+sa*Zp; Yw=Yp; Zw=-sa*Xp+ca*Zp

    # World direction -> equirectangular UV
    # Blender equirectangular: azimuth=0 at +Y axis, increases CCW (looking from above)
    # atan2(X,Z) gives azimuth measured from +Z (forward) which is +Y in Blender
    az_eq  = np.arctan2(Xw,Zw)
    el_eq  = np.arcsin(np.clip(Yw,-1,1))
    u = (az_eq/(2*np.pi)+0.5)%1.0
    v = 0.5-el_eq/np.pi

    # Bilinear sampling
    px=(u*W-0.5).clip(0,W-1); py=(v*H-0.5).clip(0,H-1)
    x0=np.floor(px).astype(int).clip(0,W-2); y0=np.floor(py).astype(int).clip(0,H-2)
    x1=x0+1; y1=y0+1
    wx=(px-x0)[...,None]; wy=(py-y0)[...,None]
    bg=(hdr[y0,x0]*(1-wx)*(1-wy) + hdr[y0,x1]*wx*(1-wy) +
        hdr[y1,x0]*(1-wx)*wy     + hdr[y1,x1]*wx*wy)

    # Exposure: Blender EV = multiply linear by 2^EV before tonemapping
    bg = bg*(2.0**exposure_ev)

    # Filmic tonemap (ACES approximation)
    bg = _aces_filmic(bg)

    return bg.astype(np.float32)  # (H,W,3) in [0,1]


def composite(pred_t, mask_t, hdr_npy, meta_row, size=256):
    """Composite relighted face over angle-correct HDRI background."""
    bg = hdri_to_background(
        hdr_npy,
        hdri_rotation_deg=float(meta_row.get("hdri_rotation_deg",0)),
        camera_yaw_deg=float(meta_row.get("camera_yaw_deg",-45)),
        exposure_ev=float(meta_row.get("exposure_used",0)),
        out_size=size)
    pred = ((pred_t[0].permute(1,2,0).cpu().float().numpy()+1)/2).clip(0,1)
    mask = mask_t[0,0].cpu().float().numpy()
    comp = pred*mask[:,:,None] + bg*(1-mask[:,:,None])
    return Image.fromarray((comp*255).clip(0,255).astype(np.uint8))

# =============================================================================
# DATASET -- mirrors training RelightDataset exactly
# =============================================================================

META_FIELDS = ["hdri_avg_lum","hdri_peak_lum","hdri_dyn_range","hdri_strength",
               "hdri_saturation","hdri_rotation_deg","exposure_used","hdri_tier"]

class RelightDataset(Dataset):
    def __init__(self,size=256,split="val",vf=0.02,seed=42):
        self.sz=size; self.df=df_meta.copy()
        self.mm,self.ms={},{}
        for f in META_FIELDS:
            if f in self.df.columns:
                v=self.df[f].astype(float)
                self.mm[f]=float(v.mean()); self.ms[f]=float(v.std())+1e-8
        self.sg={}
        for sid,g in self.df.groupby("subject_id"):
            self.sg[sid]=g.reset_index(drop=True)
        all_idx=[(s,i) for s,g in self.sg.items() for i in range(len(g))]
        r=random.Random(seed); r.shuffle(all_idx)
        nv=max(1,int(len(all_idx)*vf))
        raw=all_idx[:nv] if split=="val" else all_idx[nv:]
        self.idx=[(s,i) for s,i in raw
                  if _nk(str(self.sg[s].iloc[i]["hdri_name_key"])) in sh_index]
        self.tf=T.Compose([T.Resize((size,size)),T.ToTensor(),T.Normalize([.5]*3,[.5]*3)])

    def _sh(self,k):
        i=sh_index.get(_nk(str(k)))
        return sh_coeffs_all[i] if i is not None else np.zeros(27,dtype=np.float32)

    def _mv(self,row):
        vals=[]
        for f in META_FIELDS:
            if f=="hdri_rotation_deg":
                deg=float(row.get(f,0.0))
                vals.append(math.sin(math.radians(deg)))
                vals.append(math.cos(math.radians(deg)))
            else:
                v=(float(row.get(f,0))-self.mm.get(f,0))/self.ms.get(f,1)
                vals.append(max(-5.0,min(5.0,v)))
        return np.array(vals,dtype=np.float32)

    def _img(self,p): return self.tf(Image.open(p).convert("RGB"))

    def __len__(self): return len(self.idx)

    def __getitem__(self,idx):
        sid,i=self.idx[idx]; g=self.sg[sid]; r1=g.iloc[i]
        c=g[g["hdri_rank"]!=r1["hdri_rank"]]; r2=(c if len(c) else g).sample(1).iloc[0]
        sh2=self._sh(r2["hdri_name_key"])
        ze=torch.from_numpy(np.concatenate([sh2,self._mv(r2)]))
        return (self._img(r1["image_path"]),self._img(r2["image_path"]),
                ze,torch.tensor(int(sid),dtype=torch.long))

print("\n  Building val dataset...")
val_ds=RelightDataset(size=EVAL_CFG["image_size"],split="val",
                      vf=EVAL_CFG["val_fraction"],seed=EVAL_CFG["val_seed"])
n_eval=min(EVAL_CFG["n_eval"],len(val_ds))
val_loader=DataLoader(val_ds,batch_size=EVAL_CFG["batch_size"],
                      shuffle=False,num_workers=2,pin_memory=True)
print(f"  Val: {len(val_ds)} samples  (evaluating {n_eval})")

# =============================================================================
# METRIC HELPERS
# =============================================================================

def t2np(t): return ((t.squeeze(0).clamp(-1,1)+1)/2).permute(1,2,0).cpu().float().numpy()
def t2pil(t): return Image.fromarray((t2np(t)*255).clip(0,255).astype(np.uint8))
def psnr(p,g): m=np.mean((p-g)**2); return 100. if m<1e-10 else 10*math.log10(1/m)
def ssim(p,g): return float(_ssim_fn(p,g,data_range=1.,channel_axis=-1,win_size=7))
def pmse(p,g): return float(np.mean((p-g)**2))

# =============================================================================
# PART A -- QUANTITATIVE METRICS
# =============================================================================

print("\n"+"="*65)
print(f"  PART A -- Metrics  ({n_eval} samples, {EVAL_CFG['ddim_steps']} DDIM steps)")
print("="*65)

all_m=[]; done=0; t0=time.time()

for img1b,img2b,zeb,sidb in val_loader:
    if done>=n_eval: break
    B=min(img1b.shape[0],n_eval-done)
    i1=img1b[:B].to(DEVICE); i2=img2b[:B].to(DEVICE)
    ze=zeb[:B].to(DEVICE);   si=sidb[:B]
    with torch.no_grad():
        i1m=torch.stack([apply_mask(i1[k:k+1],si[k].item())[0] for k in range(B)])
        i2m=torch.stack([apply_mask(i2[k:k+1],si[k].item())[0] for k in range(B)])
        z1=vae_encode(i1m); z2=vae_encode(i2m)
        tr=torch.randint(0,1000,(B,),device=DEVICE)
        xt,eps=ddpm.q_sample(z2,tr)
        zev=lp3(ze); ep=unet(torch.cat([xt,z1],dim=1),tr,zev)
        lmse=F.mse_loss(ep,eps).item()
        zp=ddim_sample(z1,ze,EVAL_CFG["ddim_steps"])
        pred=vae_decode(zp)
        lp=lpips_fn(pred,i2m).squeeze()
        if lp.ndim==0: lp=lp.unsqueeze(0)
    for k in range(B):
        pn=t2np(pred[k]); gn=t2np(i2m[k])
        m={"idx":done+k,"lmse":float(lmse),"pmse":pmse(pn,gn),
           "psnr":psnr(pn,gn),"ssim":ssim(pn,gn),"lpips":float(lp[k].item())}
        all_m.append(m)
        print(f"  [{done+k+1:3d}/{n_eval}]  PSNR={m['psnr']:5.2f}  SSIM={m['ssim']:.3f}"
              f"  LPIPS={m['lpips']:.3f}  MSE={m['pmse']:.4f}  ({time.time()-t0:.0f}s)")
    done+=B

def avg(k): return float(np.mean([m[k] for m in all_m]))
def std(k): return float(np.std([m[k] for m in all_m]))

print("\n"+"="*65)
print(f"  SUMMARY  step={ckpt_step}  n={n_eval}")
print("="*65)
for name,k,hint in [
    ("Pixel MSE", "pmse","lower better"),
    ("PSNR dB",   "psnr",">20 ok  >25 good  >30 excellent"),
    ("SSIM",      "ssim",">0.70 ok  >0.85 good"),
    ("LPIPS-VGG", "lpips","lower better  <0.20 good")]:
    print(f"  {name:<12} mean={avg(k):>8.4f}  std={std(k):>7.4f}   {hint}")

pv,sv,lv=avg("psnr"),avg("ssim"),avg("lpips")
if   pv>28 and sv>0.85 and lv<0.15: verdict="STRONG -- great relighting quality"
elif pv>22 and sv>0.70 and lv<0.25: verdict="DECENT -- plausible, keep training"
elif pv>18:                          verdict="EARLY  -- needs more steps"
else:                                verdict="WEAK   -- check pipeline"
print(f"\n  {verdict}")

worst3=sorted(all_m,key=lambda m:m["lpips"],reverse=True)[:3]
worst_ids={w["idx"] for w in worst3}
print(f"\n  Worst 3 samples (highest LPIPS):")
for w in worst3:
    print(f"    s{w['idx']:03d}  PSNR={w['psnr']:.2f}  SSIM={w['ssim']:.3f}  LPIPS={w['lpips']:.3f}")

with open(EVAL_DIR/"eval_metrics.json","w") as f:
    json.dump({"ckpt":best_path.name,"step":str(ckpt_step),"n":n_eval,
               "summary":{k:{"mean":avg(k),"std":std(k)} for k in ["pmse","psnr","ssim","lpips"]},
               "samples":all_m},f,indent=2)
print(f"\n  Saved: {EVAL_DIR}/eval_metrics.json")
# =============================================================================
# PART B -- VISUAL COMPARISON GRIDS (20 grids × 32 samples each) + ZIP
# =============================================================================

print("\n"+"─"*65)
print("  PART B -- Visual Grids + ZIP")
print("─"*65)

NC        = EVAL_CFG["grid_cols"]          # 4 columns
SPG       = EVAL_CFG["samples_per_grid"]   # 32 samples per grid
N_GRIDS   = EVAL_CFG["n_grids"]            # 20 grids
NSHOW     = min(SPG * N_GRIDS, n_eval)     # total samples to render
NR        = math.ceil(SPG / NC)            # rows per grid  (32/4 = 8)

SZ = EVAL_CFG["image_size"]
PAD = 4; HDR = 22; SCH = 28

BW = SZ*3 + PAD*2 + PAD
CW = BW*NC + PAD
CH = HDR + PAD + NR*(SZ + SCH + PAD)

print(f"  Grid layout: {NC} cols × {NR} rows = {SPG} samples/grid  ×  {N_GRIDS} grids")
print(f"  Total samples to render: {NSHOW}")

try:
    fnt  = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", 9)
    fnt2 = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", 11)
except:
    fnt = fnt2 = ImageFont.load_default()

# ── Collect all samples first ────────────────────────────────────────────────
print(f"  Rendering {NSHOW} samples...")
samps = []; done2 = 0

for img1b, img2b, zeb, sidb in val_loader:
    if done2 >= NSHOW: break
    B = min(img1b.shape[0], NSHOW - done2)
    i1 = img1b[:B].to(DEVICE); i2 = img2b[:B].to(DEVICE)
    ze = zeb[:B].to(DEVICE);   si = sidb[:B]
    with torch.no_grad():
        i1m = torch.stack([apply_mask(i1[k:k+1], si[k].item())[0] for k in range(B)])
        i2m = torch.stack([apply_mask(i2[k:k+1], si[k].item())[0] for k in range(B)])
        zp  = ddim_sample(vae_encode(i1m), ze, EVAL_CFG["ddim_steps"])
        pred = vae_decode(zp)
    for k in range(B):
        samps.append((t2pil(i1m[k]), t2pil(i2m[k]), t2pil(pred[k]), all_m[done2+k]))
    done2 += B
    print(f"    Collected {done2}/{NSHOW}")

# ── Save individual sample PNGs ───────────────────────────────────────────────
SDIR = EVAL_DIR / "samples"; SDIR.mkdir(exist_ok=True)

for si2, (inp, gt, pred, m) in enumerate(samps):
    pv2 = m["psnr"]
    ind = Image.new("RGB", (SZ*3 + PAD*2, HDR + SZ + 44), (12, 12, 12))
    d2  = ImageDraw.Draw(ind)
    for li, (lb, col) in enumerate(zip(
            ["Input", "GT", "Predicted"],
            [(110,110,110), (70,190,70), (70,150,220)])):
        d2.text((li*(SZ+PAD)+2, 4), lb, fill=col, font=fnt2)
    ind.paste(inp,  (0,          HDR))
    ind.paste(gt,   (SZ+PAD,     HDR))
    ind.paste(pred, (SZ*2+PAD*2, HDR))
    sc = (65,200,65) if pv2>25 else (210,190,50) if pv2>20 else (210,65,65)
    flag = " !" if m["idx"] in worst_ids else ""
    d2.text((2, HDR+SZ+5),
            f"s{m['idx']:02d}  PSNR {pv2:.2f} dB  SSIM {m['ssim']:.3f}  LPIPS {m['lpips']:.3f}{flag}",
            fill=sc, font=fnt2)
    d2.text((2, HDR+SZ+20),
            f"pixMSE {m['pmse']:.4f}  latentMSE {m['lmse']:.4f}  step={ckpt_step}",
            fill=(55,55,55), font=fnt2)
    ind.save(SDIR / f"s{m['idx']:02d}_psnr{pv2:.1f}_ssim{m['ssim']:.3f}.png")

# ── Render one grid per 32 samples ───────────────────────────────────────────
GDIR = EVAL_DIR / "grids"; GDIR.mkdir(exist_ok=True)
grid_paths = []

for g in range(N_GRIDS):
    batch = samps[g*SPG : (g+1)*SPG]
    if not batch:
        print(f"  Grid {g+1:02d}: no samples left, stopping.")
        break

    canvas = Image.new("RGB", (CW, CH), (12, 12, 12))
    draw   = ImageDraw.Draw(canvas)

    # Column headers
    for ci in range(min(NC, len(batch))):
        xb = PAD + ci*BW
        for li, (lb, col) in enumerate(zip(
                ["Input (masked)", "Ground Truth", "Predicted"],
                [(100,100,100), (70,185,70), (70,145,215)])):
            draw.text((xb + li*(SZ+PAD) + 2, 4), lb, fill=col, font=fnt)

    for si2, (inp, gt, pred, m) in enumerate(batch):
        ri = si2 // NC; ci = si2 % NC
        xo = PAD + ci*BW
        yo = HDR + PAD + ri*(SZ + SCH + PAD)
        canvas.paste(inp,  (xo,           yo))
        canvas.paste(gt,   (xo+SZ+PAD,    yo))
        canvas.paste(pred, (xo+SZ*2+PAD*2, yo))
        pv2 = m["psnr"]
        sc  = (65,200,65) if pv2>25 else (210,190,50) if pv2>20 else (210,65,65)
        flag = " !" if m["idx"] in worst_ids else ""
        draw.text((xo+2, yo+SZ+4),
                  f"s{m['idx']:02d} PSNR{pv2:.1f} SSIM{m['ssim']:.2f} LPIPS{m['lpips']:.3f}{flag}",
                  fill=sc, font=fnt)

    gp = GDIR / f"eval_grid_{g+1:02d}.png"
    canvas.save(gp); grid_paths.append(gp)
    print(f"  Grid {g+1:02d}/{N_GRIDS} saved → {gp.name}")

# ── ZIP everything ────────────────────────────────────────────────────────────
ZP = WORK / "relight_eval.zip"
with zipfile.ZipFile(ZP, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(SDIR.glob("*.png")):
        zf.write(p, f"samples/{p.name}")
    for p in grid_paths:
        zf.write(p, f"grids/{p.name}")
    zf.write(EVAL_DIR / "eval_metrics.json", "eval_metrics.json")
print(f"\n  ZIP saved: {ZP}  ({ZP.stat().st_size/1e6:.1f} MB)  <- DOWNLOAD THIS")

# Display first grid as preview
from IPython.display import display as _D, Image as _I
if grid_paths:
    _D(_I(str(grid_paths[0]), width=min(CW, 960)))

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n"+"="*65)
print(f"  DONE  --  step={ckpt_step}")
print("="*65)
print(f"  PSNR  : {avg('psnr'):.2f} +/- {std('psnr'):.2f} dB")
print(f"  SSIM  : {avg('ssim'):.3f} +/- {std('ssim'):.3f}")
print(f"  LPIPS : {avg('lpips'):.3f} +/- {std('lpips'):.3f}")
print(f"\n  {verdict}")
print(f"\n  Output files:")
print(f"    {EVAL_DIR}/eval_grid.png")
print(f"    {EVAL_DIR}/eval_metrics.json")
print(f"    {WORK}/relight_eval.zip  <-- DOWNLOAD THIS")
print("="*65)

In [ ]:
import shutil, torch
from pathlib import Path

CKPT_DIR = Path("/kaggle/working/checkpoints")
OUT_DIR  = Path("/kaggle/working")

for fname in ["last.pt", "best.pt"]:
    src = CKPT_DIR / fname
    if not src.exists():
        src = Path("/kaggle/working/output/checkpoints") / fname
    if src.exists():
        dst = OUT_DIR / fname
        shutil.copy2(src, dst)
        ckpt = torch.load(src, map_location="cpu")
        step = ckpt.get("step", "?")
        best_loss = ckpt.get("best_loss", "?")
        print(f"✓ {fname}  ({src.stat().st_size/1e6:.1f} MB)  step={step}  best_loss={best_loss}")
    else:
        print(f"✗ {fname} not found")

print("\nDone! Download from the Output tab.")

In [1]:
# Install required packages not pre-installed on Kaggle
import subprocess, sys


def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

try:
    import lpips
except ModuleNotFoundError:
    print("Installing lpips...")
    pip_install("lpips")
    print("lpips installed ✓")

print("All dependencies ready ✓")

All dependencies ready ✓
